In [3]:
# ==================
# IMPORTS
# ==================
import requests
from bs4 import BeautifulSoup
import pandas as pd
import pandas_datareader as pdr
import time
import re
import os 
import json
import urllib.parse
from urllib.parse import quote_plus
import logging
from datetime import datetime, timedelta, date
from collections import Counter
import holidays
from requests.exceptions import RequestException, Timeout, ConnectionError as ReqConnectionError, SSLError
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from functools import wraps
import time
import logging
import random
import cloudscraper
from requests.exceptions import ConnectionError as ReqConnectionError, Timeout, SSLError
import requests
from requests.adapters import HTTPAdapter

import time
import logging
import random
import re
import cloudscraper
import pandas as pd
import threading
import atexit
import json
import os
import subprocess
from requests.exceptions import ConnectionError as ReqConnectionError, Timeout, SSLError
from threading import Lock
from collections import deque
from datetime import datetime
from requests.adapters import HTTPAdapter
from urllib.parse import urlparse, urlunparse

import os
import time
import re
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from collections import Counter
from threading import Lock


import pandas as pd
import numpy as np
import os
import logging

# Configure logging
logging.basicConfig(
    format='[%(asctime)s] %(levelname)s: %(message)s',
    level=logging.INFO
)

In [ ]:
# ==========================================
# GLOBAL CONFIGURATION
# ==========================================
TMDB_API_KEY = "0bda1399a51522fe540883215498761c"
TMDB_BASE_URL = "https://api.themoviedb.org/3"

# Shared Headers (Mimic a browser)
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Global Session  
session = requests.Session() 
session.headers.update(HEADERS)

# Create an adapter with a large pool 
adapter = HTTPAdapter(
    pool_connections=50, 
    pool_maxsize=50,      
    max_retries=2
)

# Mount the adapter to the session
session.mount('http://', adapter)
session.mount('https://', adapter)

print("✅ Global Session configured with pool_maxsize=50")

✅ Global Session configured with pool_maxsize=50


In [ ]:
# ==========================================
# LOGGING SETUP
# ==========================================
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler('bom_scraper.log'),
        logging.StreamHandler()
    ]
)

# ==========================================
# CIRCUIT BREAKER (Global Safety Switch)
# ==========================================
class CircuitBreaker:
    """Pauses ALL workers if too many 503s occur."""
    def __init__(self, failure_threshold=10, recovery_timeout=300):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.failures = 0
        self.successes = 0
        self.state = "CLOSED"
        self.last_failure_time = 0
        self.lock = Lock()
    
    def record_success(self):
        with self.lock:
            self.successes += 1
            if self.state == "HALF_OPEN" and self.successes >= 5:
                self.state = "CLOSED"
                self.failures = 0
                logging.info("✅ Circuit Breaker Recovered (CLOSED)")
            elif self.state == "CLOSED":
                self.failures = max(0, self.failures - 1)
    
    def record_failure(self):
        with self.lock:
            self.failures += 1
            self.last_failure_time = time.time()
            if self.failures >= self.failure_threshold:
                if self.state != "OPEN":
                    self.state = "OPEN"
                    logging.error(f"🔴 Circuit Breaker TRIPPED! Pausing BOM requests for {self.recovery_timeout}s")
                return True
            return False
    
    def wait_if_open(self):
        while True:
            with self.lock:
                if self.state == "CLOSED": 
                    return
                if self.state == "OPEN":
                    if time.time() - self.last_failure_time >= self.recovery_timeout:
                        self.state = "HALF_OPEN"
                        self.successes = 0
                        logging.info("🟡 Circuit Breaker Testing (HALF_OPEN)")
                        return
                # HALF_OPEN state - allow request through
                if self.state == "HALF_OPEN":
                    return
            time.sleep(5)
    
    def get_state(self):
        """Thread-safe state getter"""
        with self.lock:
            return self.state

# ==========================================
# TRACKERS & LIMITERS
# ==========================================
class FailedURLTracker:
    def __init__(self, filepath='failed_urls.json'):
        self.filepath = filepath
        self.lock = Lock()
        self.failed_urls = self._load()
    
    def _load(self):
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, 'r') as f: 
                    return json.load(f)
            except: 
                return {}
        return {}
    
    def _save(self):
        try:
            with open(self.filepath, 'w') as f: 
                json.dump(self.failed_urls, f, indent=2)
        except Exception as e:
            logging.warning(f"Failed to save failed URLs: {e}")
    
    def add(self, url, reason="unknown"):
        with self.lock:
            if url not in self.failed_urls:
                self.failed_urls[url] = {
                    'attempts': 1, 
                    'reason': reason, 
                    'time': datetime.now().isoformat()
                }
            else:
                self.failed_urls[url]['attempts'] += 1
                self.failed_urls[url]['last_attempt'] = datetime.now().isoformat()
            self._save()
    
    def remove(self, url):
        with self.lock:
            if url in self.failed_urls:
                del self.failed_urls[url]
                self._save()
    
    def get_count(self):
        with self.lock: 
            return len(self.failed_urls)
    
    def get_all(self):
        with self.lock: 
            return list(self.failed_urls.keys())

class AdaptiveRateLimiter:
    def __init__(self, initial_delay=1.5, min_delay=0.8, max_delay=10.0):
        self.delay = initial_delay
        self.min_delay = min_delay
        self.max_delay = max_delay
        self.lock = Lock()
        self.history = deque(maxlen=20)
    
    def record_response(self, status):
        with self.lock:
            self.history.append(status)
            if len(self.history) >= 10:
                success_rate = sum(1 for s in self.history if s == 200) / len(self.history)
                if success_rate > 0.9: 
                    self.delay = max(self.min_delay, self.delay * 0.95)
                elif success_rate < 0.7: 
                    self.delay = min(self.max_delay, self.delay * 1.2)
    
    def get_delay(self):
        with self.lock:
            return max(0.1, self.delay + random.uniform(-0.1, 0.2))
    
    def get_current_delay(self):
        with self.lock:
            return self.delay

# ==========================================
# CONNECTION POOL MANAGER
# ==========================================
class ConnectionPoolManager:
    def __init__(self):
        self.sessions = {}
        self.lock = Lock()
    
    def get_session(self):
        
        thread_id = threading.get_ident()
        
        with self.lock:
            if thread_id not in self.sessions:
                # Create optimized session with connection pooling
                session = cloudscraper.create_scraper(
                    browser={'browser': 'chrome', 'platform': 'windows', 'mobile': False}
                )
                # Connection Pool Adapter
                adapter = HTTPAdapter(
                    pool_connections=50, 
                    pool_maxsize=50,
                    pool_block=False,
                    max_retries=0  
                )
                session.mount('http://', adapter)
                session.mount('https://', adapter)
                
                # Force Keep-Alive to reuse TCP connections
                session.headers.update({
                    'Connection': 'keep-alive',
                    'Keep-Alive': 'timeout=60, max=1000'
                })
                
                self.sessions[thread_id] = session
                logging.info(f"🔌 Created pooled session for Thread-{thread_id}")
            
            return self.sessions[thread_id]

    def close_all(self):
        with self.lock:
            count = 0
            for sid, session in list(self.sessions.items()):
                try: 
                    session.close()
                    count += 1
                except Exception as e:
                    logging.warning(f"Error closing session {sid}: {e}")
            self.sessions.clear()
            if count > 0:
                logging.info(f"🧹 Closed {count} scraper session(s)")
    
    def get_active_count(self):
        with self.lock:
            return len(self.sessions)

# ==========================================
# GLOBAL INSTANCES
# ==========================================
circuit_breaker = CircuitBreaker()
failed_tracker = FailedURLTracker()
rate_limiter = AdaptiveRateLimiter()
connection_pool = ConnectionPoolManager()

# Ensure connections close when script exits (even on crash)
atexit.register(connection_pool.close_all)

# ==========================================
# BLOCKING REQUEST (Auto-Pooling)
# ==========================================
def blocking_request(
    url,
    params=None,
    session_obj=None,
    method="GET",
    max_retries=5,
    **kwargs
):
    # AUTOMATICALLY USE POOL if no session provided
    if session_obj is None:
        session_obj = connection_pool.get_session()
    
    if params: 
        kwargs['params'] = params
    
    is_bom = "boxofficemojo.com" in url
    is_tmdb = "themoviedb.org" in url or "api.themoviedb.org" in url
    
    # Check Circuit Breaker (BOM only)
    if is_bom: 
        circuit_breaker.wait_if_open()
    
    attempt = 0
    while attempt <= max_retries:
        try:
            # Intelligent Throttling
            if is_bom: 
                time.sleep(rate_limiter.get_delay())
            elif is_tmdb: 
                time.sleep(random.uniform(0.25, 0.35))  # TMDb rate limit
            else: 
                time.sleep(0.1)
                
            # Execute Request
            if method.upper() == "GET":
                response = session_obj.get(url, timeout=30, **kwargs)
            else:
                response = session_obj.request(method, url, timeout=30, **kwargs)
            
            status = response.status_code
            
            # ========== SUCCESS ==========
            if status == 200:
                if is_bom:
                    circuit_breaker.record_success()
                    rate_limiter.record_response(200)
                failed_tracker.remove(url)
                return response
            
            # ========== NOT FOUND ==========
            if status == 404: 
                return None
            
            # ========== RATE LIMIT ==========
            if status == 429:
                retry_after = int(response.headers.get("Retry-After", 10))
                logging.warning(f"🚦 Rate Limit 429. Pausing {retry_after}s")
                time.sleep(retry_after)
                continue
            
            # ========== SERVICE UNAVAILABLE (The BOM Nemesis) ==========
            if status == 503:
                attempt += 1
                
                if is_bom:
                    rate_limiter.record_response(503)
                    circuit_opened = circuit_breaker.record_failure()
                    
                    if circuit_opened:
                        # Wait for circuit to close
                        circuit_breaker.wait_if_open()
                        # Reset attempts after circuit recovery
                        attempt = 0
                        continue
                
                if attempt > max_retries:
                    logging.error(f"❌ 503 Persistent after {max_retries} retries: {url}")
                    failed_tracker.add(url, reason="503_persistent")
                    return None
                
                # Exponential backoff with jitter
                base_wait = 10 * (2 ** (attempt - 1))  # 10s, 20s, 40s, 80s
                jitter = random.uniform(0, base_wait * 0.3)
                total_wait = min(base_wait + jitter, 120)  # Cap at 2 minutes
                
                logging.warning(f"🛑 503 (attempt {attempt}/{max_retries}). Cooling {total_wait:.1f}s")
                time.sleep(total_wait)
                continue
            
            # ========== OTHER SERVER ERRORS ==========
            if 500 <= status < 600:
                attempt += 1
                if attempt > max_retries:
                    failed_tracker.add(url, reason=f"{status}_error")
                    return None
                
                wait = 5 * attempt
                logging.warning(f"⚠️ {status} error (attempt {attempt}). Retrying in {wait}s")
                time.sleep(wait)
                continue
            
            # ========== UNEXPECTED STATUS ==========
            logging.warning(f"⚠️ Unexpected status {status} for {url}")
            return None

        # ========== NETWORK ERRORS ===========
        except (ReqConnectionError, Timeout, SSLError) as e:
            attempt += 1
            logging.warning(f"🌐 Network error (attempt {attempt}): {type(e).__name__}")
            
            if attempt > max_retries:
                # Internet outage detection
                logging.critical("🌐 Testing network connectivity...")
                
                internet_ok = False
                for _ in range(3):
                    try:
                        import requests
                        if requests.get("https://google.com", timeout=5).status_code == 200:
                            internet_ok = True
                            break
                    except:
                        pass
                    time.sleep(5)
                
                if internet_ok:
                    # Site is down, not internet
                    logging.error(f"🌐 Site unreachable (not internet): {url}")
                    failed_tracker.add(url, reason="network_timeout")
                    return None
                else:
                    # HIBERNATION MODE
                    logging.critical("🌐 INTERNET DOWN! Entering hibernation mode...")
                    while True:
                        try:
                            import requests
                            if requests.get("https://google.com", timeout=5).status_code == 200:
                                logging.info("🌐 Internet restored!")
                                break
                        except:
                            pass
                        time.sleep(30)
                    
                    # Reset and retry
                    attempt = 0
                    continue
            
            # Exponential backoff for network errors
            time.sleep(2 ** attempt)
            continue
        
        # ========== UNEXPECTED ERRORS ==========
        except Exception as e:
            logging.error(f"⚠️ Unexpected error: {type(e).__name__} - {str(e)}")
            failed_tracker.add(url, reason=f"exception_{type(e).__name__}")
            return None
    
    # Max retries exhausted
    logging.error(f"❌ Max retries exhausted for {url}")
    failed_tracker.add(url, reason="max_retries")
    return None
     

def parse_money(money_str):
    if not money_str or not isinstance(money_str, str): 
        return 0
    
    clean_str = money_str.upper().replace(',', '').replace('$', '').strip()
    match = re.search(r'^([\d\.]+)\s*([A-Z]*)', clean_str)
    
    if not match: 
        return 0
    
    number = float(match.group(1))
    suffix = match.group(2)
    
    multiplier = 1
    if suffix in ['M', 'MILLION']: multiplier = 1_000_000
    elif suffix in ['B', 'BILLION']: multiplier = 1_000_000_000
    elif suffix in ['K', 'THOUSAND']: multiplier = 1_000
    
    return int(number * multiplier)

def get_date_id(date_obj):
    try:
        if isinstance(date_obj, str):
            date_obj = pd.to_datetime(date_obj)
        if pd.isna(date_obj):
            return 99991231  # Unknown date placeholder
        return int(date_obj.strftime('%Y%m%d'))
    except:
        return 99991231
# ==========================================
# DIAGNOSTIC TOOLS
# ==========================================
def check_port_exhaustion():
    try:
        result = subprocess.run(
            ['netstat', '-an'], 
            capture_output=True, 
            text=True, 
            timeout=5
        )
        
        time_wait_count = result.stdout.count('TIME_WAIT')
        established_count = result.stdout.count('ESTABLISHED')
        
        print(f"\n🔍 TCP Connection Status:")
        print(f"   TIME_WAIT: {time_wait_count}")
        print(f"   ESTABLISHED: {established_count}")
        
        if time_wait_count > 5000:
            print(f"   ⚠️  WARNING: High TIME_WAIT count!")
            print(f"   Consider reducing workers or increasing delays")
        elif time_wait_count < 100:
            print(f"   ✅ Connection pooling working perfectly!")
        
        return time_wait_count, established_count
    
    except Exception as e:
        logging.warning(f"Could not check port status: {e}")
        return None, None

def print_scraper_status():
    print("\n" + "="*60)
    print("📊 SCRAPER STATUS REPORT")
    print("="*60)
    print(f"Circuit Breaker State: {circuit_breaker.get_state()}")
    print(f"Circuit Failures: {circuit_breaker.failures}/{circuit_breaker.failure_threshold}")
    print(f"Current Rate Delay: {rate_limiter.get_current_delay():.2f}s")
    print(f"Failed URLs Tracked: {failed_tracker.get_count()}")
    print(f"Active Sessions: {connection_pool.get_active_count()}")
    print("="*60 + "\n")

def cleanup_sessions():
    connection_pool.close_all()

# ==========================================
# RETRY HELPER
# ==========================================
def retry_failed_urls(scraper_function, max_workers=3):
    failed_urls = failed_tracker.get_all()
    
    if not failed_urls:
        print("✅ No failed URLs to retry.")
        return []
    
    print(f"\n🔄 Retrying {len(failed_urls)} failed URLs...")
    print_scraper_status()
    
    successful = []
    failed_again = []
    
    for url in failed_urls:
        result = scraper_function(url)
        if result is not None:
            successful.append(url)
            print(f"✅ Retry succeeded: {url}")
        else:
            failed_again.append(url)
    
    print(f"\n📊 Retry Results:")
    print(f"   Success: {len(successful)}/{len(failed_urls)}")
    print(f"   Still Failed: {len(failed_again)}")
    
    return successful

In [ ]:
# ========================================== 
# PARALLEL PROCESSING UTILITIES
# ==========================================
class ParallelProcessor:
    def __init__(self, max_workers=5, rate_limit_delay=0.1):
        self.max_workers = max_workers
        self.rate_limit_delay = rate_limit_delay
        self._lock = Lock()
        self._results = {}
        self._errors = {}
        self._progress_counter = 0
        self._total_items = 0
    
    def _log_progress(self, item_id, success=True, message=""):
        with self._lock:
            self._progress_counter += 1
            status = "✅" if success else "❌"
            progress_pct = (self._progress_counter / self._total_items) * 100 if self._total_items > 0 else 0
            logging.info(f"[{self._progress_counter}/{self._total_items}] ({progress_pct:.1f}%) {status} {item_id} {message}")
    
    def _worker_wrapper(self, item, process_func, item_id_func):
        item_id = item_id_func(item) if item_id_func else str(item)
        
        try:
            # Add rate limiting delay
            #time.sleep(self.rate_limit_delay)
            # Process the item
            result = process_func(item)
            
            if result is not None:
                with self._lock:
                    self._results[item_id] = result
                self._log_progress(item_id, success=True)
                return (item_id, result, None)
            else:
                with self._lock:
                    self._errors[item_id] = "No result returned"
                self._log_progress(item_id, success=False, message="(no result)")
                return (item_id, None, "No result returned")
                
        except Exception as e:
            error_msg = str(e)
            with self._lock:
                self._errors[item_id] = error_msg
            self._log_progress(item_id, success=False, message=f"({error_msg[:50]})")
            logging.error(f"Error processing {item_id}: {error_msg}")
            return (item_id, None, error_msg)
    
    def process_batch(self, items, process_func, item_id_func=None, desc="Processing"):
        self._results = {}
        self._errors = {}
        self._progress_counter = 0
        self._total_items = len(items)
        
        if not items:
            logging.info(f"{desc}: No items to process")
            return {}
        
        logging.info(f"🚀 {desc}: Starting parallel processing of {len(items)} items with {self.max_workers} workers")
        
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = {
                executor.submit(self._worker_wrapper, item, process_func, item_id_func): item 
                for item in items
            }
            
            # Wait for all futures to complete
            for future in as_completed(futures):
                try:
                    future.result()  # Raise any exceptions from the worker
                except Exception as e:
                    item = futures[future]
                    item_id = item_id_func(item) if item_id_func else str(item)
                    logging.error(f"Future failed for {item_id}: {e}")
        
        success_count = len(self._results)
        error_count = len(self._errors)
        logging.info(f"✅ {desc}: Completed! {success_count} succeeded, {error_count} failed")
        
        return self._results
    
    def get_errors(self):
        return self._errors.copy()


def parallel_fetch(items, fetch_func, id_func=None, max_workers=5, rate_limit=0.1, desc="Fetching"):
    processor = ParallelProcessor(max_workers=max_workers, rate_limit_delay=rate_limit)
    return processor.process_batch(items, fetch_func, id_func, desc)


def parallel_process_movies(movie_rows, process_func, id_col='movie_id', max_workers=5, desc="Processing movies"):
    processor = ParallelProcessor(max_workers=max_workers, rate_limit_delay=0.15)
    
    results = processor.process_batch(
        items=movie_rows,
        process_func=process_func,
        item_id_func=lambda row: row.get(id_col, 'unknown'),
        desc=desc
    )
    
    # Return as list in original order where possible
    output = []
    for row in movie_rows:
        movie_id = row.get(id_col)
        if movie_id in results:
            output.append(results[movie_id])
    
    return output


def parallel_process(items, process_func, max_workers=5, desc="Processing"):
    if not items:
        logging.info(f"{desc}: No items to process")
        return []
    
    total = len(items)
    results = []
    completed = 0
    
    logging.info(f"🚀 {desc}: Starting parallel processing of {total} items with {max_workers} workers")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_item = {executor.submit(process_func, item): item for item in items}
        
        # Collect results as they complete
        for future in as_completed(future_to_item):
            item = future_to_item[future]
            try:
                result, message = future.result()
                results.append((result, message))
                completed += 1
                
                # Log progress every 10 items or at completion
                if completed % 10 == 0 or completed == total:
                    pct = (completed / total) * 100
                    logging.info(f"   [{completed}/{total}] ({pct:.1f}%) {desc}")
                    
            except Exception as e:
                completed += 1
                error_msg = str(e)[:100]
                logging.error(f"Error processing item: {error_msg}")
                results.append((None, f"ERROR: {error_msg}"))
    
    success_count = sum(1 for r, _ in results if r is not None)
    logging.info(f"✅ {desc}: Completed! {success_count}/{total} succeeded")
    
    return results

In [ ]:
# ==========================================
# INFLATION DATA MANAGER (CSV-BASED)
# ==========================================
INFLATION_CSV = "inflation_data.csv"

def create_inflation_csv():
    print("📊 [Inflation] Fetching CPI data from FRED...")
    try:
        # Get CPI data from 1970 to present (covers classic films like The Omen 1976)
        start = datetime(1950, 1, 1)
        monthly_cpi = pdr.get_data_fred('CPIAUCSL', start, datetime.now())
        
        # Convert to DataFrame with year/month columns
        inflation_df = monthly_cpi.reset_index()
        inflation_df.columns = ['date', 'cpi']
        inflation_df['year'] = inflation_df['date'].dt.year
        inflation_df['month'] = inflation_df['date'].dt.month
        
        # Calculate inflation factor (adjust to latest CPI)
        latest_cpi = inflation_df['cpi'].iloc[-1]
        inflation_df['inflation_factor'] = latest_cpi / inflation_df['cpi']
        
        # Select and order columns
        inflation_df = inflation_df[['year', 'month', 'cpi', 'inflation_factor']]
        
        # Save to CSV
        inflation_df.to_csv(INFLATION_CSV, index=False)
        
        print(f"   ✅ Inflation CSV created: {len(inflation_df)} months")
        print(f"   📅 Date range: {inflation_df['year'].min()}-{inflation_df['month'].min():02d} to {inflation_df['year'].max()}-{inflation_df['month'].max():02d}")
        print(f"   📈 Latest CPI: {latest_cpi:.2f}")
        
        return inflation_df
        
    except Exception as e:
        print(f"   ❌ Failed to fetch CPI data: {e}")
        return None

def load_inflation_data():
    """Load inflation data from CSV, creating it if needed."""
    if not os.path.exists(INFLATION_CSV):
        print(f"   📝 {INFLATION_CSV} not found. Creating...")
        df = create_inflation_csv()
        if df is None:
            print("   ⚠️ Using nominal values (no inflation adjustment)")
            return None
    else:
        try:
            df = pd.read_csv(INFLATION_CSV)
            print(f"   ✅ Inflation data loaded: {len(df)} months")
        except Exception as e:
            print(f"   ❌ Failed to load {INFLATION_CSV}: {e}")
            return None
    
    # Create lookup dictionary for fast access: 
    inflation_lookup = {}
    for _, row in df.iterrows():
        key = (int(row['year']), int(row['month']))
        inflation_lookup[key] = row['inflation_factor']
    
    return inflation_lookup

def adjust_for_inflation(amount, release_date, inflation_lookup):
    if not amount or amount == 0 or inflation_lookup is None:
        return amount
    
    if pd.isna(release_date):
        return amount
    
    try:
        # Parse date
        if isinstance(release_date, str):
            release_date = pd.to_datetime(release_date)
        
        year = release_date.year
        month = release_date.month
         
        # Lookup inflation factor
        factor = inflation_lookup.get((year, month))
        
        if factor is None:
            return amount  # Date outside CPI range
        
        return int(amount * factor)
        
    except:
        return amount

# Initialize global inflation lookup
print("🔧 Initializing inflation data...")
INFLATION_LOOKUP = load_inflation_data()

In [ ]:
# ==========================================
# GLOBAL GENRE MAP
# ==========================================
def fetch_global_genre_map():
    print("🎭 [GenreMap] Fetching genre list from TMDb...")
    url = f"{TMDB_BASE_URL}/genre/movie/list"
    resp = blocking_request(url, params={'api_key': TMDB_API_KEY}, session_obj=session)
    
    if resp:
        try:
            genres = resp.json().get('genres', [])
            genre_map = {g['id']: g['name'] for g in genres}
            print(f"   ✅ Loaded {len(genre_map)} genres from TMDb")
            return genre_map
        except Exception as e:
            print(f"   ⚠️ Failed to fetch genres: {e}")
    
    # Fallback to empty dict if API fails
    print("   ⚠️ Using genre IDs as fallback")
    return {}

# Initialize global genre map (shared across all builders)
GENRE_MAP = fetch_global_genre_map()

In [8]:
# ==========================================
# GLOBAL CHECKPOINTING SYSTEM
# ==========================================
def load_checkpoint(filename, id_col='movie_id'):

    if not os.path.exists(filename):
        print(f"   📝 No checkpoint found. Starting fresh.")
        return pd.DataFrame(), set()
    
    try:
        df = pd.read_csv(filename)
        if df.empty:
            print(f"   📝 Empty checkpoint found. Starting fresh.")
            return pd.DataFrame(), set()
        
        # Extract IDs of movies that were successfully saved
        if id_col in df.columns:
            processed_ids = set(df[id_col].dropna().unique())
            print(f"   ✅ Checkpoint loaded: {len(df)} rows, {len(processed_ids)} unique {id_col}s already saved")
            return df, processed_ids
        else:
            print(f"   ⚠️ Checkpoint missing '{id_col}' column. Starting fresh.")
            return pd.DataFrame(), set()
    except Exception as e:
        print(f"   ⚠️ Failed to load checkpoint: {e}. Starting fresh.")
        return pd.DataFrame(), set()

def save_checkpoint(new_rows, existing_df, filename):

    if not new_rows:
        return existing_df
    
    try:
        new_df = pd.DataFrame(new_rows)
        
        # Combine old + new data
        if existing_df.empty:
            combined = new_df
        else:
            combined = pd.concat([existing_df, new_df], ignore_index=True)
        
        # Atomic save: write to temp file first, then rename
        temp_file = filename + '.tmp'
        combined.to_csv(temp_file, index=False)
        
        # Replace old file with new one (atomic on most systems)
        if os.path.exists(filename):
            os.remove(filename)
        os.rename(temp_file, filename)
        
        print(f"   💾 Checkpoint saved: {len(new_rows)} new rows | Total: {len(combined)} rows")
        
        return combined
    except Exception as e:
        print(f"   ⚠️ Checkpoint save failed: {e}")
        return existing_df

def is_already_processed(movie_id, processed_ids):

    return movie_id in processed_ids

In [ ]:
# ===============================
# BOM url normalizer
# ===============================
def normalize_bom_release_url(bom_url: str) -> str:
    if not bom_url:
        return bom_url

    p = urlparse(bom_url)

    # Keep only the path, strip daily/weekly/weekend suffixes
    clean_path = re.sub(r'/(daily|weekly|weekend)/?$', '/', p.path)
    clean_path = clean_path.rstrip('/')

    # Rebuild URL without query or fragment
    return urlunparse((p.scheme, p.netloc, clean_path, '', '', ''))


In [ ]:
# =================================================
# REVENUE AND THEATER HELPER: EXTRACTION LOGIC 
# =================================================
def extract_bom_data(soup, bom_url):
    first_week_gross = None
    opening_weekend_gross = None
    theater_count = None
    
    try:
        page_text = soup.get_text()
        
        # Foreign Currency Safety Check
        if re.search(r'[€£¥₹₩]', page_text[:3000]):
            return None, None, None
        
        # GET OPENING WEEKEND (From Text Summary) 
        opening_patterns = [
            r'(?<!Widest )Opening[:\s]*\$([\d]{1,3}(?:,\d{3})*)',
            r'Opening\s*Weekend[:\s]*\$([\d]{1,3}(?:,\d{3})*)',
            r'Domestic\s*Opening[:\s]*\$?([\d]{1,3}(?:,\d{3})*)',
        ]
        for pattern in opening_patterns:
            match = re.search(pattern, page_text, re.IGNORECASE)
            if match:
                val = int(match.group(1).replace(',', ''))
                # Sanity Check: Must be > 3000
                if val > 3000 and val < 600_000_000:
                    opening_weekend_gross = val
                    break
        
        # EXTRACT THEATER COUNT (Opening first, then Widest fallback)
        opening_theater_count = None
        widest_theater_count = None

        text_content = soup.get_text()

        # 1) Try to find opening theater count from “Opening: $... NNN theaters”
        theater_after_opening = re.search(
            r'(?<!Widest\s)Opening[:\s]*\$[\d]{1,3}(?:,\d{3})*[^\d]*([\d]{1,2},\d{3}|[\d]{1,4})\s*theaters?',
            text_content,
            re.IGNORECASE
        )
        if theater_after_opening:
            try:
                count = int(theater_after_opening.group(1).replace(',', ''))
                if 1 <= count <= 10000:
                    opening_theater_count = count
            except (ValueError, AttributeError):
                pass

        # 2) DOM-aware fallback: look near elements mentioning “opening”
        if not opening_theater_count:
            summary_divs = soup.find_all(['div', 'span', 'td', 'th'])
            for i, elem in enumerate(summary_divs):
                elem_text = elem.get_text().strip().lower()
                if 'opening' in elem_text and 'widest' not in elem_text:
                    # Scan a small window of following elements for “… NNN theaters”
                    for j in range(i, min(i + 10, len(summary_divs))):
                        sibling_text = summary_divs[j].get_text()
                        theater_match = re.search(r'([\d,]+)\s*theaters?', sibling_text, re.IGNORECASE)
                        if theater_match:
                            try:
                                count = int(theater_match.group(1).replace(',', ''))
                                if 1 <= count <= 10000:
                                    opening_theater_count = count
                                    break
                            except (ValueError, AttributeError):
                                continue
                    if opening_theater_count:
                        break

        # 3) Widest release fallback
        if not opening_theater_count:
            widest_patterns = [
                r'Widest\s*Release[:\s]*([\d,]+)',
                r'widest[:\s]*([\d,]+)',
            ]
            for pattern in widest_patterns:
                match = re.search(pattern, text_content, re.IGNORECASE)
                if match:
                    try:
                        count = int(match.group(1).replace(',', ''))
                        if 1 <= count <= 10000:
                            widest_theater_count = count
                            break
                    except (ValueError, AttributeError):
                        continue

        # 4) Final assignment: prefer opening, then widest
        if opening_theater_count is not None:
            theater_count = opening_theater_count
        elif widest_theater_count is not None:
            theater_count = widest_theater_count
        # else: theater_count stays as None

        # Normalizer
        base_url = normalize_bom_release_url(bom_url) if bom_url else None
       
        # Weekly and Weekend Tables Extraction
        # === STRATEGY 1: WEEKLY TABLE (Strict $ and >3000) ===
        if base_url:
            weekly_url = base_url + '/weekly/'
            time.sleep(0.3)
            resp = blocking_request(weekly_url)

            if resp and resp.status_code == 200:
                w_soup = BeautifulSoup(resp.content, 'html.parser')
                tables = w_soup.find_all('table')

                for table in tables:
                    rows = table.find_all('tr')
                    if len(rows) < 2: continue
                
                    header_cells = rows[0].find_all(['th', 'td'])
                    data_cells = rows[1].find_all('td')
                
                    weekly_col_idx = -1
                    for idx, cell in enumerate(header_cells):
                        if 'weekly' in cell.get_text().lower():
                            weekly_col_idx = idx
                            break
                
                    found_val = None
                
                    if weekly_col_idx != -1 and len(data_cells) > weekly_col_idx:
                        txt = data_cells[weekly_col_idx].get_text().strip()
                        if '$' in txt: 
                            val = parse_money(txt)
                            if val and val > 3000: found_val = val
                
                    if not found_val:
                        for cell in data_cells:
                            txt = cell.get_text().strip()
                            if '$' in txt:
                                val = parse_money(txt)
                                if val and val > 3000:
                                    found_val = val
                                    break 
                
                    if found_val:
                        first_week_gross = found_val
                        break

        # === STRATEGY 2: WEEKEND TABLE (Strict Math) ===
        if base_url and (not first_week_gross or opening_weekend_gross is None):
            weekend_url = base_url + '/weekend/'
            time.sleep(0.3)
            resp = blocking_request(weekend_url)
            
            if resp and resp.status_code == 200:
                we_soup = BeautifulSoup(resp.content, 'html.parser')
                tables = we_soup.find_all('table')
                
                for table in tables:
                    rows = table.find_all('tr')
                    if len(rows) < 3: continue 
                    
                    header_cells = rows[0].find_all(['th', 'td'])
                    to_date_idx = -1
                    weekend_idx = -1
                    
                    for idx, cell in enumerate(header_cells):
                        txt = cell.get_text().lower()
                        if 'to date' in txt:
                            to_date_idx = idx
                        elif 'weekend' in txt and weekend_idx == -1:
                            weekend_idx = idx
                    
                    if to_date_idx != -1 and weekend_idx != -1:
                        if opening_weekend_gross is None and len(rows)>1:
                            first_raw_cells = rows[1].find_all('td')
                            if len(first_raw_cells) > weekend_idx:
                                ow_txt = first_raw_cells[weekend_idx].get_text().strip()
                                if '$' in ow_txt:
                                    ow_val = parse_money(ow_txt)
                                    if ow_val and ow_val > 3000 and ow_val < 600_000_000:
                                        opening_weekend_gross = ow_val
                        
                        if not first_week_gross and len(rows) > 2:
                            data_cells = rows[2].find_all('td')
                            if len(data_cells) > max(to_date_idx, weekend_idx):
                                td_txt = data_cells[to_date_idx].get_text().strip()
                                we_txt = data_cells[weekend_idx].get_text().strip()
                            
                                if '$' in td_txt and '$' in we_txt:
                                    td_val = parse_money(td_txt)
                                    we_val = parse_money(we_txt)
                                    if (td_val and td_val > 3000) and (we_val and we_val > 3000):
                                        first_week_gross = td_val - we_val
                                        break

        # === STRATEGY 3: DAILY TABLE (Sum 1st 7) ===
        if not first_week_gross:
            tables = soup.find_all('table')
            for table in tables:
                rows = table.find_all('tr')
                if len(rows) < 8: continue
                
                header_cells = rows[0].find_all(['th', 'td'])
                daily_col_idx = -1
                
                for idx, cell in enumerate(header_cells):
                    if 'daily' in cell.get_text().lower():
                        daily_col_idx = idx
                        break
                
                if daily_col_idx == -1:
                    test_row = rows[1].find_all('td')
                    for idx, cell in enumerate(test_row):
                        if '$' in cell.get_text():
                            daily_col_idx = idx
                            break
                
                if daily_col_idx != -1:
                    daily_sum = 0
                    count = 0
                    for i in range(1, 8):
                        if i >= len(rows): break
                        cells = rows[i].find_all('td')
                        if len(cells) > daily_col_idx:
                            txt = cells[daily_col_idx].get_text().strip()
                            if '$' in txt:
                                val = parse_money(txt)
                                if val and val > 3000:
                                    daily_sum += val
                                    count += 1
                    
                    if count >= 3:
                        first_week_gross = daily_sum
                        break

        # Fallback Estimate
        if not first_week_gross and opening_weekend_gross:
            first_week_gross = int(opening_weekend_gross * 2.3)
            
    except Exception as e:
        pass
        
    return first_week_gross, opening_weekend_gross, theater_count

In [ ]:
# ==========================================
# CONFIGURATION 
# ==========================================
START_YEAR = 2000
END_YEAR = 2025  
LIMIT_PER_YEAR = 2000  
MIN_THEATERS = 0     
MIN_RUNTIME = 0        
MIN_BUDGET = 0
MIN_BUDGET_ANIMATION = 0

# Keywords indicating source material (for is_based_on_source)
SOURCE_KEYWORDS = { 
    'based on book', 'based on novel', 'based on comic', 'marvel comics',
    'dc comics', 'based on graphic novel', 'based on play',
    'based on true story', 'based on short story', 'based on video game',
    'biography', 'based on real events', 'comic book', 'superhero',
    'remake', 'reboot', 'adaptation',
    'marvel cinematic universe', 'dc extended universe', 'based on cartoon',
    'based on tv series', 'based on anime', 'based on manga',
    'literary adaptation', 'video game adaptation'
}

OUTPUT_FILE = "Movie_Dimension.csv"

In [ ]:
# ==========================================
# MOVIE DIMENSION COLLECTOR 
# ==========================================

class MovieDimensionCollector:
    def __init__(self, max_workers=5):
        self.bom_cache = {}
        self.max_workers = max_workers
        self._cache_lock = Lock()  # Thread-safe cache access
        print(f"✓ Movie Dimension Collector initialized with {max_workers} parallel workers")
    
    # --- 1. BOM: Get Movie List ---
    def get_yearly_releases_bom(self, year, max_movies=2000):
        movies = []
        seen_titles = set()
        
        # ==============================================================================
        # STEP 1: Get ALL releases from BOM Monthly Calendar
        # ==============================================================================
        print(f"   🔍 Scraping BOM Monthly Calendar for {year}...")
        
        # Iterate through all 12 months
        for month in range(1, 13):
            # BOM calendar URL format: /calendar/YYYY-MM-01/
            calendar_url = f"https://www.boxofficemojo.com/calendar/{year}-{month:02d}-01/"
            
            time.sleep(0.3)  
            response = blocking_request(calendar_url)
            
            if not response or response.status_code != 200:
                print(f"   ⚠️ Failed to fetch calendar for {year}-{month:02d}")
                continue
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Find all movie release links in the calendar table
            month_count = 0
            
            for link in soup.find_all('a', href=True):
                href = link.get('href', '')
                
                # Only process release links
                if '/release/' not in href:
                    continue
                
                title = link.get_text().strip()
                if not title:
                    continue
                
                # Deduplicate
                title_normalized = title.lower().strip()
                if title_normalized in seen_titles:
                    continue
                seen_titles.add(title_normalized)
                
                # Build full URL
                full_url = "https://www.boxofficemojo.com" + href if not href.startswith('http') else href
                
                movies.append({
                    'title': title,
                    'bom_url': full_url,
                    'year': year,
                    'source': 'BOM'
                })
                month_count += 1
                
                if len(movies) >= max_movies:
                    break
            
            print(f"   📅 {year}-{month:02d}: Found {month_count} releases")
            
            if len(movies) >= max_movies:
                break
        
        bom_count = len(movies)
        print(f"   📋 BOM Calendar: Found {bom_count} total releases")
        
        # ==============================================================================
        # STEP 2: Supplement with TMDb 
        # ==============================================================================
        if len(movies) < max_movies:
            remaining = max_movies - len(movies)
            print(f"   🔍 TMDb: Fetching additional movies to reach target...")
            
            page = 1
            max_pages = 10 
            tmdb_added = 0
            
            while tmdb_added < remaining and page <= max_pages:
                discover_url = f"{TMDB_BASE_URL}/discover/movie"
                params = {
                    'api_key': TMDB_API_KEY,
                    'primary_release_year': year,
                    'sort_by': 'revenue.desc',
                    'with_release_type': '2|3',
                    'region': 'US',
                    'page': page,
                    'vote_count.gte': 10 
                }
                
                resp = blocking_request(discover_url, params=params)
                if not resp:
                    break
                
                try:
                    results = resp.json().get('results', [])
                    if not results:
                        break
                    
                    for movie in results:
                        title = movie.get('title', '')
                        if not title:
                            continue
                        
                        title_normalized = title.lower().strip()
                        if title_normalized in seen_titles:
                            continue
                        seen_titles.add(title_normalized)
                        
                        # Create BOM search URL
                        search_title = urllib.parse.quote(title)
                        bom_url = f"https://www.boxofficemojo.com/search/?q={search_title}"
                        
                        movies.append({
                            'title': title,
                            'bom_url': bom_url,
                            'year': year,
                            'source': 'TMDb',
                            'tmdb_id': movie.get('id')
                        })
                        tmdb_added += 1
                        
                        if tmdb_added >= remaining:
                            break
                    
                    page += 1
                    time.sleep(0.1)
                
                except Exception as e:
                    logging.warning(f"TMDb discover error: {e}")
                    break
            
            print(f"   📋 TMDb: Added {tmdb_added} additional movies")
        
        total = len(movies)
        print(f"   📋 Total: {total} movies to process")
        return movies[:max_movies]
    
    def get_tmdb_data(self, title, year):
        # Search
        search_url = f"{TMDB_BASE_URL}/search/movie"
        params = {'api_key': TMDB_API_KEY, 'query': title, 'year': year}
        response = blocking_request(search_url, params=params)
        
        if not response:
            return None
        
        results = response.json().get('results', [])
        if not results:
            return None
        
        movie_id = results[0]['id']
        
        # Get full details
        details_url = f"{TMDB_BASE_URL}/movie/{movie_id}"
        details_resp = blocking_request(details_url, params={'api_key': TMDB_API_KEY})
        
        if not details_resp:
            return None
        
        return details_resp.json()
    
    # --- 2. Box Office Mojo: DOMESTIC Weekly/Opening Data ---
    def get_bom_box_office_data(self, bom_url):
        # Thread-safe cache check
        with self._cache_lock:
            if bom_url in self.bom_cache:
                return self.bom_cache[bom_url]
        
        first_week_gross = None
        opening_weekend_gross = None
        theater_count = None
        days_collected = 0
        
        try:
            # Handle search URLs 
            if '/search/' in bom_url:
                time.sleep(0.3)
                search_resp = blocking_request(bom_url)
                if not search_resp or search_resp.status_code != 200:
                    with self._cache_lock:
                        self.bom_cache[bom_url] = (None, None, None, 0)
                    return None, None, None, 0
                
                search_soup = BeautifulSoup(search_resp.content, 'html.parser')
                # Find first release link in search results
                release_link = search_soup.find('a', href=lambda h: h and '/release/' in h)
                if not release_link:
                    with self._cache_lock:
                        self.bom_cache[bom_url] = (None, None, None, 0)
                    return None, None, None, 0
                
                # Update to actual release URL
                bom_url = "https://www.boxofficemojo.com" + release_link['href']
            
            # Get movie page
            time.sleep(0.3)
            response = blocking_request(bom_url)
            if not response or response.status_code != 200:
                with self._cache_lock:
                    self.bom_cache[bom_url] = (None, None, None, 0)
                return None, None, None, 0
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # === Find DOMESTIC (US) release page 
            domestic_link = None
            
            # Strategy 1: Look for "Domestic" in release selector/links
            for link in soup.find_all('a', href=True):
                href = link.get('href', '')
                link_text = link.get_text().lower()
                if '/release/' in href and ('domestic' in link_text or 'united states' in link_text or 'u.s.' in link_text):
                    domestic_link = href
                    break
            
            # Strategy 2: Check page content - if showing non-USD currency, try to find USD version
            page_text = soup.get_text()
            showing_foreign_currency = bool(re.search(r'[€£¥₹₩]|CNY|EUR|GBP|JPY|KRW|INR', page_text))
            
            if showing_foreign_currency and not domestic_link:
                # Try appending domestic area code to URL
                if '?' in bom_url:
                    domestic_url_try = bom_url + '&area=XNA'
                else:
                    domestic_url_try = bom_url + '?area=XNA'
                
                test_resp = blocking_request(domestic_url_try)
                if test_resp and test_resp.status_code == 200:
                    test_soup = BeautifulSoup(test_resp.content, 'html.parser')
                    test_text = test_soup.get_text()
                    if '$' in test_text and not re.search(r'[€£¥₹₩]', test_text[:2000]):
                        soup = test_soup
                        bom_url = domestic_url_try
            
            # Navigate to domestic link if found
            if domestic_link:
                if not domestic_link.startswith('http'):
                    domestic_link = "https://www.boxofficemojo.com" + domestic_link
                time.sleep(0.2)
                dom_resp = blocking_request(domestic_link)
                if dom_resp and dom_resp.status_code == 200:
                    soup = BeautifulSoup(dom_resp.content, 'html.parser')
                    bom_url = domestic_link
            
            # Get text content
            text_content = soup.get_text()
            
            # === FINAL USD VALIDATION ===
            if re.search(r'[€£¥₹₩]', text_content[:3000]):
                with self._cache_lock:
                    self.bom_cache[bom_url] = (None, None, None, 0)
                return None, None, None, 0
            
            # === Use helper function to get BOM data ===
            first_week_gross, opening_weekend_gross, theater_count = extract_bom_data(soup, bom_url)

            days_collected = 7 if first_week_gross else 0

        
        except Exception as e:
            pass
        
        result = (first_week_gross, opening_weekend_gross, theater_count, days_collected)
        with self._cache_lock:
            self.bom_cache[bom_url] = result
        return result
    
    # --- 3. TMDb Details + Keywords ---
    def get_tmdb_full_details(self, movie_id):
        url = f"{TMDB_BASE_URL}/movie/{movie_id}"
        params = {'api_key': TMDB_API_KEY}
        response = blocking_request(url, params=params)
        
        if not response:
            return None
        
        data = response.json()
        
        # Get keywords for source material detection
        is_based_on_source = 0
        keywords_url = f"{TMDB_BASE_URL}/movie/{movie_id}/keywords"
        keywords_resp = blocking_request(keywords_url, params=params)
        
        if keywords_resp:
            keywords_data = keywords_resp.json().get('keywords', [])
            keyword_names = {k['name'].lower() for k in keywords_data}
            if any(sk in keyword_names for sk in SOURCE_KEYWORDS):
                is_based_on_source = 1
        
        data['is_based_on_source'] = is_based_on_source
        
        return data
    
    # --- 4. MPAA Rating from TMDb (US Certification) ---
    def get_mpaa_rating(self, movie_id):
        if not movie_id:
            return None
        
        url = f"{TMDB_BASE_URL}/movie/{movie_id}/release_dates"
        response = blocking_request(url, params={'api_key': TMDB_API_KEY})
        
        if response:
            try:
                data = response.json()
                results = data.get('results', [])
                
                # Find US release info
                for country in results:
                    if country.get('iso_3166_1') == 'US':
                        release_dates = country.get('release_dates', [])
                        # Look for theatrical releases with certification
                        for release in release_dates:
                            cert = release.get('certification', '')
                            if cert and cert.strip():
                                return cert
                        break
            except:
                pass
        
        return None
    
    # --- 5. Count Competing Movies ---
    def count_competing_movies(self, movie_id, release_date, fact_rows_so_far):
        #Count movies in the fact table that released within ±3 days
        if not release_date or not fact_rows_so_far:
            return 0
        
        try:
            target_date = release_date
            window_start = target_date - timedelta(days=3)
            window_end = target_date + timedelta(days=3)
            
            count = 0
            for row in fact_rows_so_far:
                if row['movie_id'] == movie_id:
                    continue
                
                row_date_str = row.get('release_date')
                if row_date_str:
                    try:
                        row_date = datetime.strptime(row_date_str, '%Y-%m-%d')
                        if window_start <= row_date <= window_end:
                            count += 1
                    except:
                        continue
            
            return count
        except:
            return 0
    
    # --- 6. Process Single Movie (for parallel execution) ---
    def _process_single_movie(self, movie_data):
        #Process a single movie for parallel execution
        movie = movie_data['movie']
        year = movie_data['year']
        processed_ids = movie_data['processed_ids']
        current_run_ids = movie_data['current_run_ids']
        
        title = movie['title']
        
        # Get TMDb data FIRST to check release date before anything else
        tmdb_data = self.get_tmdb_data(title, year)
        if not tmdb_data:
            return None, f"{title}: ❌ No TMDb match"
        
        movie_id = tmdb_data['id']
        
        # Check if already processed (from checkpoint OR from current run)
        # Thread-safe check with lock
        with self._cache_lock:
            if is_already_processed(movie_id, processed_ids) or movie_id in current_run_ids:
                return None, f"{title}: ⏭️ Already processed"
        
        # EARLY release date validation
        release_date_str = tmdb_data.get('release_date', '')
        try:
            r_date = datetime.strptime(release_date_str, '%Y-%m-%d')
            date_id = int(r_date.strftime('%Y%m%d'))
        except:
            return None, f"{title}: ❌ Invalid Date"
        
        # Filter by date range EARLY
        start_date = datetime(START_YEAR, 1, 1)
        end_date = datetime(END_YEAR, 12, 31)
        if r_date < start_date or r_date > end_date:
            return None, f"{title}: ⏭️ Released {r_date.year} (outside {START_YEAR}-{END_YEAR})"
        
        # Get BOM data
        fw_gross, ow_gross, theaters, days_collected = self.get_bom_box_office_data(movie['bom_url'])
        
        if not fw_gross:
            return None, f"{title}: ❌ No BOM data ({days_collected}d)"
        
        budget = tmdb_data.get('budget', 0)
        runtime = tmdb_data.get('runtime', 0)
        imdb_id = tmdb_data.get('imdb_id')
        original_language = tmdb_data.get('original_language')
        
        # Check keywords for source material
        is_based_on_source = 0
        keywords_url = f"{TMDB_BASE_URL}/movie/{movie_id}/keywords"
        keywords_resp = blocking_request(keywords_url, params={'api_key': TMDB_API_KEY})
        if keywords_resp:
            try:
                keywords = keywords_resp.json().get('keywords', [])
                keyword_names = {k['name'].lower() for k in keywords}
                is_based_on_source = 1 if any(kw in keyword_names for kw in SOURCE_KEYWORDS) else 0
            except:
                pass
        
        # Get genres
        genres = tmdb_data.get('genres', [])
        genre_names = [g['name'] for g in genres]
        genre_string = '|'.join(genre_names) if genre_names else None
        genre_count = len(genre_names)
        
        # Get production countries
        production_countries = tmdb_data.get('production_countries', [])
        country_names = [c['name'] for c in production_countries]
        countries_string = '|'.join(country_names) if country_names else None
        country_count = len(country_names)
        
        # Determine if US film
        country_codes = [c.get('iso_3166_1', '').upper() for c in production_countries]
        country_names_lower = [c['name'].lower() for c in production_countries]
        is_us_film = 1 if ('US' in country_codes or 
                          'united states of america' in country_names_lower or
                          'united states' in country_names_lower or
                          'usa' in country_names_lower) else 0
        
        # Check if Animation
        is_animation = any(g.get('id') == 16 for g in genres)
        
        # Calculate inflation-adjusted budget EARLY for filtering
        budget_adjusted_for_filter = adjust_for_inflation(budget, r_date, INFLATION_LOOKUP) if budget else 0
        
        # Apply budget filter using INFLATION-ADJUSTED budget
        min_budget_threshold = MIN_BUDGET_ANIMATION if is_animation else MIN_BUDGET
        if budget_adjusted_for_filter < min_budget_threshold or runtime < MIN_RUNTIME:
            return None, f"{title}: ⏭️ Skipped (Adj Budget ${budget_adjusted_for_filter/1e6:.0f}M or Runtime {runtime}m)"
        
        # Theater count filter (Unused for 0 minimum range)
        if theaters is None:
            return None, f"{title}: ⏭️ Skipped (Theater count N/A)"
        elif theaters < MIN_THEATERS:
            return None, f"{title}: ⏭️ Skipped (Limited release: {theaters} theaters)"
        
        # Get MPAA rating
        mpaa_rating = self.get_mpaa_rating(movie_id)
        
        # Placeholder for competition count
        num_competing = 0
        
        # Calculate inflation-adjusted values
        fw_gross_adjusted = adjust_for_inflation(fw_gross, r_date, INFLATION_LOOKUP) if fw_gross else 0
        ow_gross_adjusted = adjust_for_inflation(ow_gross, r_date, INFLATION_LOOKUP) if ow_gross else 0
        budget_adjusted = adjust_for_inflation(budget, r_date, INFLATION_LOOKUP) if budget else 0
    
        # Determine budget class
        budget_millions = budget / 1_000_000
        if is_animation:
            if budget_millions < 30:
                budget_class = 'mid_budget'
            elif budget_millions < 100:
                budget_class = 'high_budget'
            else:
                budget_class = 'mega_budget'
        else:  
            if budget_millions < 50:
                budget_class = 'mid_budget'
            elif budget_millions < 150:
                budget_class = 'high_budget'
            else:
                budget_class = 'mega_budget'
    
        # Assemble fact row
        fact_row = {
        'movie_id': movie_id,
        'title': title,
        'imdb_id': imdb_id,
        'original_language': original_language,
        'genres': genre_string,
        'genre_count': genre_count,
        'production_countries': countries_string,
        'country_count': country_count,
        'is_us_film': is_us_film,
        'mpaa_rating': mpaa_rating,
        'is_based_on_source': is_based_on_source,
        'num_competing_movies': num_competing,
        'date_id': date_id,
        'release_date': r_date.strftime('%Y-%m-%d'),
        'collection_id': tmdb_data['belongs_to_collection']['id'] if tmdb_data.get('belongs_to_collection') else None,
        'budget_raw': budget,
        'budget_adjusted': int(budget_adjusted),
        'budget_class': budget_class,
        'revenue_first_week_raw': fw_gross,
        'revenue_first_week_adjusted': int(fw_gross_adjusted),
        'revenue_opening_weekend_raw': ow_gross if ow_gross else 0,
        'revenue_opening_weekend_adjusted': int(ow_gross_adjusted) if ow_gross else 0,
        'first_week_days_collected': days_collected,
        'opening_theater_count': theaters,
        'runtime_minutes': runtime
        }
    
        fw_millions = fw_gross / 1_000_000
        return fact_row, f"{title}: ✅ ${fw_millions:.1f}M / {theaters} theaters"

    # --- MAIN PIPELINE ---
    def build_fact_table(self):
        """Build fact table with TMDb metadata + Box Office Mojo weekly data + movie dimension features."""
        existing_df, processed_ids = load_checkpoint(OUTPUT_FILE, id_col='movie_id')
    
        existing_rows = existing_df.to_dict('records') if not existing_df.empty else []
    
        # Track movie_ids added during THIS run to prevent duplicates across BOM years
        current_run_ids = set()
    
        print(f"🎬 Starting Fact Table Generation ({START_YEAR}-{END_YEAR}) - PARALLEL with {self.max_workers} workers...")
        print(f"   Criteria: Budget >= ${MIN_BUDGET:,}, Runtime >= {MIN_RUNTIME}m")
        if processed_ids:
            print(f"   ⏭️ Skipping {len(processed_ids)} already processed movies")
    
        all_fact_rows = []
    
        for year in range(START_YEAR, END_YEAR + 1):
            print(f"\n📅 Processing Year: {year}")
            bom_movies = self.get_yearly_releases_bom(year, max_movies=LIMIT_PER_YEAR)
        
            # Prepare items for parallel processing
            items_to_process = []
            for movie in bom_movies:
                items_to_process.append({
                    'movie': movie,
                    'year': year,
                    'processed_ids': processed_ids,
                    'current_run_ids': current_run_ids
                })
        
            if not items_to_process:
                print(f"   No movies to process for {year}")
                continue
        
            # Process movies in parallel
            results = parallel_process(
                items=items_to_process,
                process_func=self._process_single_movie,
                max_workers=self.max_workers,
                desc=f"Year {year}"
            )
        
            # Collect results and update current_run_ids
            year_count = 0
            skip_reasons = {
                'no_tmdb': 0,
                'already_processed': 0,
                'invalid_date': 0,
                'outside_range': 0,
                'no_bom_data': 0,
                'low_budget_runtime': 0,
                'limited_release': 0,
                'added': 0
            }
        
            for fact_row, message in results:
                # Log each movie's result
                logging.info(message)
            
                if fact_row is not None:
                    # Thread-safe update of current_run_ids
                    with self._cache_lock:
                        if fact_row['movie_id'] not in current_run_ids:
                            current_run_ids.add(fact_row['movie_id'])
                            all_fact_rows.append(fact_row)
                            year_count += 1
                            skip_reasons['added'] += 1
                else:
                    # Categorize skip reason for summary
                    if "No TMDb" in message:
                        skip_reasons['no_tmdb'] += 1
                    elif "Already processed" in message:
                        skip_reasons['already_processed'] += 1
                    elif "Invalid Date" in message:
                        skip_reasons['invalid_date'] += 1
                    elif "outside" in message:
                        skip_reasons['outside_range'] += 1
                    elif "No BOM" in message:
                        skip_reasons['no_bom_data'] += 1
                    elif "Budget" in message or "Runtime" in message:
                        skip_reasons['low_budget_runtime'] += 1
                    elif "Limited release" in message or "Theater count" in message:
                         skip_reasons['limited_release'] += 1
        
            # Print year summary
            print(f"   --> Year {year} complete:")
            print(f"       ✅ Added: {skip_reasons['added']}")
            if skip_reasons['no_tmdb'] > 0:
                print(f"       ❌ No TMDb match: {skip_reasons['no_tmdb']}")
            if skip_reasons['already_processed'] > 0:
                print(f"       ⏭️ Already processed: {skip_reasons['already_processed']}")
            if skip_reasons['no_bom_data'] > 0:
                print(f"       ❌ No BOM data: {skip_reasons['no_bom_data']}")
            if skip_reasons['low_budget_runtime'] > 0:
                print(f"       ⏭️ Low budget/runtime: {skip_reasons['low_budget_runtime']}")
            if skip_reasons['limited_release'] > 0:
                print(f"       ⏭️ Limited release (<600 theaters): {skip_reasons['limited_release']}")
            if skip_reasons['outside_range'] > 0:
                print(f"       ⏭️ Outside date range: {skip_reasons['outside_range']}")
        
            # Checkpoint after each year
            if all_fact_rows:
                existing_df = save_checkpoint(all_fact_rows, existing_df, OUTPUT_FILE)
                existing_rows.extend(all_fact_rows)
                all_fact_rows = []
    
        # Final save
        if all_fact_rows:
            existing_df = save_checkpoint(all_fact_rows, existing_df, OUTPUT_FILE)
    
        # POST-PROCESSING: Recalculate competition counts for ALL movies
        print(f"\n🔄 Recalculating competition counts")
        all_rows = existing_df.to_dict('records')
    
        for i, row in enumerate(all_rows):
            try:
                movie_id = row['movie_id']
                release_date_str = row['release_date']
                release_date = datetime.strptime(release_date_str, '%Y-%m-%d')
            
                window_start = release_date - timedelta(days=3)
                window_end = release_date + timedelta(days=3)
               
                count = 0
                for other_row in all_rows:
                    if other_row['movie_id'] == movie_id:
                        continue
                
                    other_date_str = other_row.get('release_date')
                    if other_date_str:
                        try:
                            other_date = datetime.strptime(other_date_str, '%Y-%m-%d')
                            if window_start <= other_date <= window_end:
                                count += 1
                        except:
                            continue
            
                all_rows[i]['num_competing_movies'] = count
            except:
                all_rows[i]['num_competing_movies'] = 0
    
        # Save updated dataframe
        existing_df = pd.DataFrame(all_rows)
        existing_df.to_csv(OUTPUT_FILE, index=False)
        print(f"   ✅ Competition counts updated")
    
        print(f"\n🎉 Completed! Fact table saved to {OUTPUT_FILE}")
        print(f"   Total Movies: {len(existing_df)}")
        if not existing_df.empty:
            print(existing_df.head())
#Instantiate and run
if __name__ == "__main__":
    collector = MovieDimensionCollector(max_workers=5)
    collector.build_fact_table()

In [13]:
# ==========================================
# CONFIGURATION
# ==========================================
INPUT_FILE = "Movie_Dimension.csv"
OUTPUT_FILE = "Dim_Date.csv"

In [ ]:
# ==========================================
# DATE DIMENSION BUILDER
# ==========================================
class DateDimensionBuilder:
    def __init__(self):
        self.us_holidays = None # Initialized after we know the year range

    def get_year_range(self):
        
        if not os.path.exists(INPUT_FILE):
            print(f"❌ {INPUT_FILE} not found. Cannot determine date range.")
            return None, None
            
        df = pd.read_csv(INPUT_FILE)
        
        # Filter out valid dates 
        valid_dates = df[df['date_id'] != 99991231]['date_id']
        
        if valid_dates.empty:
            print("⚠️ No valid dates found in Fact Table. Defaulting to 2020-2025.")
            return 2020, 2025

        # Convert to datetime to extract year safely
        dates = pd.to_datetime(valid_dates.astype(str), format='%Y%m%d')
        
        min_year = dates.dt.year.min()
        max_year = dates.dt.year.max()
        
        return int(min_year), int(max_year)

    def is_holiday_release_window(self, target_date):
        
        try:
            # Check 7 days ahead (inclusive of release day)
            for i in range(7):
                check_day = target_date + timedelta(days=i)
                if check_day in self.us_holidays:
                    return 1
            return 0
        except:
            return 0

    def build_dataset(self):
        # 1. Determine Range
        min_year, max_year = self.get_year_range()
        if not min_year: return

        print(f"📅 Generating Date Dimension for years {min_year}-{max_year}...")

        # 2. Initialize Holidays for this range
        # Add a buffer year at the end to ensure the 7-day lookahead works for Dec 31st releases
        self.us_holidays = holidays.US(years=range(min_year, max_year + 2))
        
        # 3. Generate Continuous Date Range
        start_date = date(min_year, 1, 1)
        end_date = date(max_year, 12, 31)
        delta = end_date - start_date
        
        date_rows = []
        
        for i in range(delta.days + 1):
            current_date = start_date + timedelta(days=i)
            
            # --- BASIC FEATURE ENGINEERING ---
            
            # Basic Calendar
            year = current_date.year
            month = current_date.month
            day = current_date.day
            day_of_week = current_date.weekday() # 0=Monday, 6=Sunday
            
            # Seasons 
            # Summer: May (5), June (6), July (7), August (8)
            is_summer = 1 if month in [5, 6, 7, 8] else 0
            
            # Holiday Season: November (11), December (12)
            is_holiday_season = 1 if month in [11, 12] else 0
            
            # Pandemic Release (2020-2022 inclusive)
            is_pandemic_release = 1 if 2020 <= year <= 2022 else 0
            
            # Weekend (Friday=4, Saturday=5, Sunday=6) 
            is_weekend = 1 if day_of_week >= 4 else 0
            
            # Holiday Release Window (7-day lookahead)
            is_holiday_release = self.is_holiday_release_window(current_date)
            
            row = {
                'date_id': int(current_date.strftime('%Y%m%d')),
                'date_iso': current_date.strftime('%Y-%m-%d'),
                'year': year,
                'quarter': (month - 1) // 3 + 1,
                'month': month,
                'month_name': current_date.strftime('%B'),
                'day_of_month': day,
                'day_of_week': day_of_week,
                'day_name': current_date.strftime('%A'),
                'is_weekend': is_weekend,
                'is_summer_season': is_summer,
                'is_holiday_season': is_holiday_season,
                'is_pandemic_release': is_pandemic_release,
                'is_holiday_release': is_holiday_release
            }
            
            date_rows.append(row)

        # 4. Save
        df = pd.DataFrame(date_rows)
        df.to_csv(OUTPUT_FILE, index=False)
        print(f"🎉 Saved {OUTPUT_FILE}")
        print(f"   Total Dates: {len(df)}")
        print(f"   Range: {df['date_iso'].min()} to {df['date_iso'].max()}")
        print("   Sample Data:")
        print(df.head())

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    builder = DateDimensionBuilder()
    builder.build_dataset()

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
INPUT_FILE = "Movie_Dimension.csv"
OUTPUT_FILE = "Dim_Franchise.csv"

# Keywords that indicate a franchise even if collection_id is missing
FRANCHISE_KEYWORDS = {
    'sequel', 'prequel', 'spin-off', 'remake', 'reboot',
    'marvel cinematic universe', 'dc extended universe', 'mcu', 'dceu',
    'shared universe', 'cinematic universe',
    'franchise', 'trilogy',
    'part 2', 'part ii', 'part 3', 'part iii', 'part iv', 'part v',
    'continuation', 'follow-up'
}

In [ ]:
# ==========================================
# FRANCHISE HISTORY BUILDER
# ==========================================

class FranchiseHistoryBuilder:
    def __init__(self, max_workers=5):
        self.keywords_cache = {}  # Cache keyword lookups
        self.collection_cache = {}  # Cache collection data
        self.bom_cache = {}  # Cache BOM box office data
        self.imdb_cache = {}  # Cache TMDb -> IMDb ID mapping
        self.budget_cache = {}  # Cache TMDb budget data (in USD)
        self.max_workers = max_workers
        self._cache_lock = Lock()  # Thread-safe cache access
        print(f"✓ Franchise builder initialized with {max_workers} parallel workers")

    def get_imdb_id(self, tmdb_movie_id):
        with self._cache_lock:
            if tmdb_movie_id in self.imdb_cache:
                return self.imdb_cache[tmdb_movie_id]
        
        try:
            time.sleep(0.1)  # Rate limiting for TMDb API
            url = f"{TMDB_BASE_URL}/movie/{tmdb_movie_id}/external_ids"
            resp = blocking_request(url, params={'api_key': TMDB_API_KEY}, session_obj=session)
            
            if resp and resp.status_code == 200:
                data = resp.json()
                imdb_id = data.get('imdb_id')
                with self._cache_lock:
                    self.imdb_cache[tmdb_movie_id] = imdb_id
                return imdb_id
        except:
            pass
        
        with self._cache_lock:
            self.imdb_cache[tmdb_movie_id] = None
        return None

    def get_movie_budget(self, tmdb_movie_id):
        with self._cache_lock:
            if tmdb_movie_id in self.budget_cache:
                return self.budget_cache[tmdb_movie_id]
        
        try:
            time.sleep(0.1)  # Rate limiting for TMDb API
            url = f"{TMDB_BASE_URL}/movie/{tmdb_movie_id}"
            resp = blocking_request(url, params={'api_key': TMDB_API_KEY}, session_obj=session)
            
            if resp and resp.status_code == 200:
                data = resp.json()
                budget = data.get('budget', 0)
                # TMDb uses 0 for unknown budget, treat as None
                budget = budget if budget > 0 else None
                with self._cache_lock:
                    self.budget_cache[tmdb_movie_id] = budget
                return budget
        except:
            pass
        
        with self._cache_lock:
            self.budget_cache[tmdb_movie_id] = None
        return None

    def get_bom_data_for_movie(self, tmdb_movie_id, title, year):
        cache_key = f"{tmdb_movie_id}_{year}"
        with self._cache_lock:
            if cache_key in self.bom_cache:
                return self.bom_cache[cache_key]
        
        first_week_gross = None
        opening_weekend_gross = None
        bom_url = None
        
        try:
            # === STRATEGY 1: Use IMDb ID for direct BOM lookup ===
            imdb_id = self.get_imdb_id(tmdb_movie_id)
            if imdb_id:
                # BOM uses IMDb ID in URLs: /title/{imdb_id}/
                bom_url = f"https://www.boxofficemojo.com/title/{imdb_id}/"
                response = blocking_request(bom_url)
                
                if response and response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'html.parser')
                    # Check if we got a valid page 
                    if 'Box Office' in soup.get_text() or 'Opening' in soup.get_text():
                        first_week_gross, opening_weekend_gross = self._extract_bom_data(soup, bom_url)
            
            # === STRATEGY 2: Fall back to year page title search ===
            if not first_week_gross:
                year_url = f"https://www.boxofficemojo.com/year/{year}/"
                response = blocking_request(year_url)
                
                if response and response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'html.parser')
                    
                    # Find movie link by title
                    found_url = None
                    rows = soup.find_all('tr')
                    for row in rows[1:]:
                        link_tag = row.find('a', href=True)
                        if not link_tag:
                            continue
                        
                        href = link_tag['href']
                        if '/release/' not in href:
                            continue
                        
                        link_title = link_tag.get_text().strip().lower()
                        search_title = title.lower()
                        
                        # Fuzzy match
                        if search_title in link_title or link_title in search_title:
                            found_url = "https://www.boxofficemojo.com" + href
                            break
                    
                    if found_url:
                        time.sleep(0.3)
                        response = blocking_request(found_url)
                        if response and response.status_code == 200:
                            soup = BeautifulSoup(response.content, 'html.parser')
                            first_week_gross, opening_weekend_gross = self._extract_bom_data(soup, found_url)
        
        except Exception as e:
            pass
        
        result = (first_week_gross, opening_weekend_gross)
        with self._cache_lock:
            self.bom_cache[cache_key] = result
        return result

    def _extract_bom_data(self, soup, bom_url=None):
        first_week_gross = None
        opening_weekend_gross = None
        
        try:
            # === STEP 1: Find and navigate to DOMESTIC release page ===
            domestic_link = None
            release_link = None
            
            # Strategy 1: Look for any /release/ link on the page
            for link in soup.find_all('a', href=True):
                href = link.get('href', '')
                link_text = link.get_text().lower().strip()
                
                if '/release/' in href:
                    # Check parent and grandparent for "domestic" or "opening" context
                    context_text = ''
                    parent = link.parent
                    if parent:
                        context_text += parent.get_text().lower()
                        grandparent = parent.parent
                        if grandparent:
                            context_text += ' ' + grandparent.get_text().lower()
                    
                    # Check for domestic indicators in context
                    if 'domestic' in context_text or 'domestic opening' in context_text:
                        domestic_link = href
                        break
                    elif '$' in link_text or re.match(r'\$[\d,]+', link_text):
                        if not release_link:
                            release_link = href
            
            if not domestic_link and release_link:
                domestic_link = release_link
            
            # Strategy 2: Check releasegroup links for domestic
            if not domestic_link:
                release_links = soup.find_all('a', href=lambda h: h and '/releasegroup/' in h)
                for link in release_links:
                    if 'domestic' in link.get_text().lower():
                        domestic_link = link.get('href')
                        break
            
            # Strategy 3: Check for foreign currency and try area=XNA
            page_text = soup.get_text()
            showing_foreign_currency = bool(re.search(r'[€£¥₹₩]|CNY|EUR|GBP|JPY|KRW|INR', page_text))
            
            if showing_foreign_currency and not domestic_link and bom_url:
                if '?' in bom_url:
                    domestic_url_try = bom_url + '&area=XNA'
                else:
                    domestic_url_try = bom_url + '?area=XNA'
                
                test_resp = blocking_request(domestic_url_try)
                if test_resp and test_resp.status_code == 200:
                    test_soup = BeautifulSoup(test_resp.content, 'html.parser')
                    test_text = test_soup.get_text()
                    if '$' in test_text and not re.search(r'[€£¥₹₩]', test_text[:2000]):
                        soup = test_soup
                        bom_url = domestic_url_try
            
            # Navigate to domestic link if found
            if domestic_link:
                if not domestic_link.startswith('http'):
                    domestic_link = "https://www.boxofficemojo.com" + domestic_link
                # Remove /weekend or /weekly from URL to get base page
                domestic_link = re.sub(r'/(weekend|weekly).*$', '/', domestic_link)
                time.sleep(0.2)
                dom_resp = blocking_request(domestic_link)
                if dom_resp and dom_resp.status_code == 200:
                    soup = BeautifulSoup(dom_resp.content, 'html.parser')
                    bom_url = domestic_link
            
            # Get text content
            text_content = soup.get_text()
            
            # === FINAL USD VALIDATION ===
            if re.search(r'[€£¥₹₩]', text_content[:3000]):
                return None, None
            
            # Use the helper function
            fw, ow, _ = extract_bom_data(soup, bom_url)
            first_week_gross = fw
            opening_weekend_gross = ow
        except Exception:
            return None, None
        
        return first_week_gross, opening_weekend_gross

    def check_keywords_for_franchise(self, movie_id):
        with self._cache_lock:
            if movie_id in self.keywords_cache:
                return self.keywords_cache[movie_id]
        time.sleep(0.1)  # Rate limiting for TMDb API
        url = f"{TMDB_BASE_URL}/movie/{movie_id}/keywords"
        resp = blocking_request(url, params={'api_key': TMDB_API_KEY}, session_obj=session)
        
        if not resp:
            result = {'is_shared_universe': False}
            with self._cache_lock:
                self.keywords_cache[movie_id] = result
            return result
        
        try:
            keywords = resp.json().get('keywords', [])
            names = {str(k.get('name') or '').lower() for k in keywords}
            
            # Check for shared universe indicators
            is_shared_universe = any(kw in names for kw in [
                # Marvel/DC
                'marvel cinematic universe', 'mcu', 
                'dc extended universe', 'dceu', 'dc universe',
                # General universe terms
                'shared universe', 'cinematic universe', 'expanded universe',
                # Monster/Horror universes
                'monsterverse', 'conjuring universe', 'dark universe',
                # Fantasy universes
                'wizarding world', 'middle-earth', 'narnia',
                # Sci-Fi universes
                'star wars', 'star trek', 'alien universe', 'predator',
                # Sony Spider-Man Universe
                'spider-man universe', 'sony spider-man universe', 'spider-verse', 'venom',
                'sony pictures universe of marvel characters', 'spumc',
                # Transformers Universe
                'transformers', 'transformers universe', 'transformers cinematic universe',
                # Action universes
                'fast and furious', 'furious 7', 'the fast saga', 'fast x',
                'john wick', 'mission impossible',
                # Animation universes
                'pixar', 'dreamworks', 'illumination',
                # Other major universes
                'jurassic world', 'jurassic park', 'godzilla', 'kong',
                'x-men', 'avatar', 'planet of the apes', 'terminator',
                'matrix', 'ghostbusters', 'men in black'
            ])
            
            result = {'is_shared_universe': is_shared_universe}
            with self._cache_lock:
                self.keywords_cache[movie_id] = result
            return result
        except:
            result = {'is_shared_universe': False}
            with self._cache_lock:
                self.keywords_cache[movie_id] = result
            return result

    def get_collection_history(self, collection_id, target_release_date_id):
        if not collection_id or pd.isna(collection_id):
            return []

        # Check collection cache first
        with self._cache_lock:
            if collection_id in self.collection_cache:
                parts = self.collection_cache[collection_id]
            else:
                parts = None
        
        if parts is None:
            url = f"{TMDB_BASE_URL}/collection/{int(collection_id)}"
            resp = blocking_request(url, params={'api_key': TMDB_API_KEY}, session_obj=session)
            
            if not resp: 
                return []
            
            try:
                data = resp.json()
                parts = data.get('parts', [])
                with self._cache_lock:
                    self.collection_cache[collection_id] = parts
            except:
                return []
        
        history = []
        
        # Parse target date
        try:
            target_date_str = str(int(target_release_date_id))
            target_date = pd.to_datetime(target_date_str, format='%Y%m%d')
        except:
            return []

        for part in parts:
            r_date_str = part.get('release_date', '')
            if not r_date_str: 
                continue
            
            try:
                r_date = pd.to_datetime(r_date_str)
                
                # Only include prior movies
                if r_date.date() < target_date.date():
                    movie_id = part.get('id')
                    title = part.get('title', '')
                    year = r_date.year
                    
                    # Get BOM data using TMDb movie ID (for IMDb lookup) + title fallback
                    first_week_raw, opening_weekend_raw = self.get_bom_data_for_movie(movie_id, title, year)
                    
                    # Get budget from TMDb
                    budget_raw = self.get_movie_budget(movie_id)
                    
                    # Track if data is MISSING (None) vs just 0
                    opening_weekend_missing = opening_weekend_raw is None
                    budget_missing = budget_raw is None
                    
                    # Include with 0 values to maintain franchise continuity
                    first_week_raw = first_week_raw if first_week_raw else 0
                    opening_weekend_raw = opening_weekend_raw if opening_weekend_raw else 0
                    budget_raw = budget_raw if budget_raw else 0
                    
                    # Adjust for inflation
                    first_week_adj = adjust_for_inflation(first_week_raw, r_date, INFLATION_LOOKUP) if first_week_raw > 0 else 0
                    opening_weekend_adj = adjust_for_inflation(opening_weekend_raw, r_date, INFLATION_LOOKUP) if opening_weekend_raw > 0 else 0
                    budget_adj = adjust_for_inflation(budget_raw, r_date, INFLATION_LOOKUP) if budget_raw > 0 else 0
                    
                    history.append({
                        'title': title,
                        'release_date': r_date,
                        'first_week_gross_raw': first_week_raw,
                        'first_week_gross_adjusted': int(first_week_adj),
                        'opening_weekend_raw': opening_weekend_raw,
                        'opening_weekend_adjusted': int(opening_weekend_adj),
                        'budget_raw': budget_raw,
                        'budget_adjusted': int(budget_adj),
                        'has_missing_data': opening_weekend_missing,
                        'has_missing_budget': budget_missing
                    })
            except:
                continue
        
        # Sort: Newest first (L1 is the most recent predecessor)
        history.sort(key=lambda x: x['release_date'], reverse=True)
        return history

    def build_dataset(self):
        if not os.path.exists(INPUT_FILE):
            print(f"❌ {INPUT_FILE} not found. Run Fact Table Generator first.")
            return

        # Load checkpoint
        existing_df, processed_ids = load_checkpoint(OUTPUT_FILE, id_col='movie_id')
        
        fact_df = pd.read_csv(INPUT_FILE)
        
        # Filter out already processed and invalid movies
        movies_to_process = []
        for idx, row in fact_df.iterrows():
            movie_id = row['movie_id']
            date_id = row['date_id']
            
            if date_id == 99991231:
                continue
            if is_already_processed(movie_id, processed_ids):
                continue
            
            movies_to_process.append(row.to_dict())
        
        print(f"🎬 Processing Franchise History for {len(movies_to_process)} movies (parallel)...")
        if processed_ids:
            print(f"   ⏭️ Skipping {len(processed_ids)} already processed movies")
        
        if not movies_to_process:
            print("   No new movies to process.")
            return
        
        def process_single_movie(row):
            movie_id = row['movie_id']
            date_id = row['date_id']
            collection_id = row.get('collection_id')
            
            # 1. Get franchise history
            history = []
            has_collection = pd.notna(collection_id) and collection_id != 0
            
            # Check if movie is part of a shared universe
            keyword_info = self.check_keywords_for_franchise(movie_id)
            is_shared_universe = 1 if keyword_info.get('is_shared_universe', False) else 0
            
            # Get collection history
            if has_collection:
                history = self.get_collection_history(int(collection_id), date_id)
            
            # 2. Determine if it's a franchise
            prior_count = len(history)
            
            if prior_count > 0:
                is_franchise = 1
            else:
                is_franchise = 0
            
            # 3. Calculate Aggregates
            avg_first_week_raw = 0
            avg_first_week_adj = 0
            sum_first_week_raw = 0
            sum_first_week_adj = 0
            max_first_week_raw = 0
            max_first_week_adj = 0
            last_first_week_raw = 0
            last_first_week_adj = 0
            avg_opening_weekend_raw = 0
            avg_opening_weekend_adj = 0
            sum_opening_weekend_raw = 0
            sum_opening_weekend_adj = 0
            max_opening_weekend_raw = 0
            max_opening_weekend_adj = 0
            last_opening_weekend_raw = 0
            last_opening_weekend_adj = 0
            avg_budget_raw = 0
            avg_budget_adj = 0
            max_budget_raw = 0
            max_budget_adj = 0
            last_budget_raw = 0
            last_budget_adj = 0
            years_diff = 0
            movies_missing_data = 0
            movies_missing_budget = 0
            
            if prior_count > 0:
                # First week aggregates
                first_week_raw_values = [h['first_week_gross_raw'] for h in history if h['first_week_gross_raw'] > 0]
                first_week_adj_values = [h['first_week_gross_adjusted'] for h in history if h['first_week_gross_adjusted'] > 0]
                if first_week_raw_values:
                    avg_first_week_raw = int(sum(first_week_raw_values) / len(first_week_raw_values))
                    sum_first_week_raw = int(sum(first_week_raw_values))
                    max_first_week_raw = int(max(first_week_raw_values))
                if first_week_adj_values:
                    avg_first_week_adj = int(sum(first_week_adj_values) / len(first_week_adj_values))
                    sum_first_week_adj = int(sum(first_week_adj_values))
                    max_first_week_adj = int(max(first_week_adj_values))
                last_first_week_raw = int(history[0]['first_week_gross_raw'])
                last_first_week_adj = int(history[0]['first_week_gross_adjusted'])
                
                # Opening weekend aggregates
                opening_raw_values = [h['opening_weekend_raw'] for h in history if h['opening_weekend_raw'] > 0]
                opening_adj_values = [h['opening_weekend_adjusted'] for h in history if h['opening_weekend_adjusted'] > 0]
                if opening_raw_values:
                    avg_opening_weekend_raw = int(sum(opening_raw_values) / len(opening_raw_values))
                    sum_opening_weekend_raw = int(sum(opening_raw_values))
                    max_opening_weekend_raw = int(max(opening_raw_values))
                if opening_adj_values:
                    avg_opening_weekend_adj = int(sum(opening_adj_values) / len(opening_adj_values))
                    sum_opening_weekend_adj = int(sum(opening_adj_values))
                    max_opening_weekend_adj = int(max(opening_adj_values))
                last_opening_weekend_raw = int(history[0]['opening_weekend_raw'])
                last_opening_weekend_adj = int(history[0]['opening_weekend_adjusted'])
                
                # Budget aggregates
                budget_raw_values = [h['budget_raw'] for h in history if h['budget_raw'] > 0]
                budget_adj_values = [h['budget_adjusted'] for h in history if h['budget_adjusted'] > 0]
                if budget_raw_values:
                    avg_budget_raw = int(sum(budget_raw_values) / len(budget_raw_values))
                    max_budget_raw = int(max(budget_raw_values))
                if budget_adj_values:
                    avg_budget_adj = int(sum(budget_adj_values) / len(budget_adj_values))
                    max_budget_adj = int(max(budget_adj_values))
                last_budget_raw = int(history[0]['budget_raw'])
                last_budget_adj = int(history[0]['budget_adjusted'])
                
                last_film_date = history[0]['release_date']
                try:
                    target_date = pd.to_datetime(str(int(date_id)), format='%Y%m%d')
                    diff_days = (target_date - last_film_date).days
                    years_diff = round(diff_days / 365.25, 2)
                except:
                    years_diff = 0
                
                movies_missing_data = sum(1 for h in history if h.get('has_missing_data', False))
                movies_missing_budget = sum(1 for h in history if h.get('has_missing_budget', False))
            
            # 4. Build Row
            row_data = {
                'movie_id': int(movie_id),
                'collection_id': int(collection_id) if pd.notna(collection_id) else None,
                'is_franchise': is_franchise,
                'is_shared_universe': is_shared_universe,
                'franchise_prior_movies_count': prior_count,
                'franchise_movies_missing_data': movies_missing_data,
                'franchise_movies_missing_budget': movies_missing_budget,
                # First week features
                'franchise_avg_prior_first_week_raw': avg_first_week_raw,
                'franchise_avg_prior_first_week_adj': avg_first_week_adj,
                'franchise_sum_prior_first_week_raw': sum_first_week_raw,
                'franchise_sum_prior_first_week_adj': sum_first_week_adj,
                'franchise_max_prior_first_week_raw': max_first_week_raw,
                'franchise_max_prior_first_week_adj': max_first_week_adj,
                'franchise_last_film_first_week_raw': last_first_week_raw,
                'franchise_last_film_first_week_adj': last_first_week_adj,
                # Opening weekend features
                'franchise_avg_prior_opening_weekend_raw': avg_opening_weekend_raw,
                'franchise_avg_prior_opening_weekend_adj': avg_opening_weekend_adj,
                'franchise_sum_prior_opening_weekend_raw': sum_opening_weekend_raw,
                'franchise_sum_prior_opening_weekend_adj': sum_opening_weekend_adj,
                'franchise_max_prior_opening_weekend_raw': max_opening_weekend_raw,
                'franchise_max_prior_opening_weekend_adj': max_opening_weekend_adj,
                'franchise_last_film_opening_weekend_raw': last_opening_weekend_raw,
                'franchise_last_film_opening_weekend_adj': last_opening_weekend_adj,
                # Budget features
                'franchise_avg_prior_budget_raw': avg_budget_raw,
                'franchise_avg_prior_budget_adj': avg_budget_adj,
                'franchise_max_prior_budget_raw': max_budget_raw,
                'franchise_max_prior_budget_adj': max_budget_adj,
                'franchise_last_film_budget_raw': last_budget_raw,
                'franchise_last_film_budget_adj': last_budget_adj,
                'years_since_last_franchise_film': years_diff
            }
            
            # 5. Unroll History (L1 to L10)
            for i in range(10):
                prefix = f"L{i+1}"
                if i < len(history):
                    film = history[i]
                    row_data[f"{prefix}_title"] = film['title']
                    row_data[f"{prefix}_release_date"] = film['release_date'].strftime('%Y-%m-%d')
                    row_data[f"{prefix}_first_week_raw"] = int(film['first_week_gross_raw'])
                    row_data[f"{prefix}_first_week_adj"] = int(film['first_week_gross_adjusted'])
                    row_data[f"{prefix}_opening_weekend_raw"] = int(film['opening_weekend_raw'])
                    row_data[f"{prefix}_opening_weekend_adj"] = int(film['opening_weekend_adjusted'])
                else:
                    row_data[f"{prefix}_title"] = None
                    row_data[f"{prefix}_release_date"] = None
                    row_data[f"{prefix}_first_week_raw"] = 0
                    row_data[f"{prefix}_first_week_adj"] = 0
                    row_data[f"{prefix}_opening_weekend_raw"] = 0
                    row_data[f"{prefix}_opening_weekend_adj"] = 0
            
            return row_data
        
        # Use parallel processing
        results = parallel_fetch(
            items=movies_to_process,
            fetch_func=process_single_movie,
            id_func=lambda row: row['movie_id'],
            max_workers=self.max_workers,
            rate_limit=0.15,
            desc="Franchise History"
        )
        
        # Collect results and save
        output_rows = list(results.values())
        
        if output_rows:
            existing_df = save_checkpoint(output_rows, existing_df, OUTPUT_FILE)
        
        print(f"\n🎉 Completed! Dim_Franchise saved to {OUTPUT_FILE}")
        print(f"   Total Rows: {len(existing_df)}")

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    builder = FranchiseHistoryBuilder(max_workers=5)
    builder.build_dataset()

In [86]:
# ==========================================
# CONFIGURATION
# ==========================================
INPUT_FILE = "Movie_Dimension.csv"
OUTPUT_FILE = "Dim_Director.csv"

In [ ]:
# ==========================================
# DIRECTOR HISTORY BUILDER 
# ==========================================
class DirectorHistoryBuilder:
    def __init__(self, max_workers=5):
        self.director_cache = {}       # Cache director credits (filmography)
        self.movie_credits_cache = {}  # Cache movie credits (to find director)
        self.movie_details_cache = {}  # Cache movie details (budget, runtime)     
        self.max_workers = max_workers
        self._cache_lock = Lock()      # Thread-safe cache access
        print(f"✓ Director builder initialized with {max_workers} parallel workers")

    def get_credits_for_movie(self, movie_id):
        cache_key = f"credits_{movie_id}"
        
        with self._cache_lock:
            if cache_key in self.movie_credits_cache:
                return self.movie_credits_cache[cache_key]
        
        url = f"{TMDB_BASE_URL}/movie/{movie_id}/credits"
        resp = blocking_request(url, params={'api_key': TMDB_API_KEY}, session_obj=session)
        if not resp:
            return None
        
        try:
            credits = resp.json()
            with self._cache_lock:
                self.movie_credits_cache[cache_key] = credits
            return credits
        except:
            return None

    def get_director_for_movie(self, movie_id):
        credits = self.get_credits_for_movie(movie_id)
        if not credits:
            return None, None, 0, None
        
        try:
            crew = credits.get('crew', [])
        except:
            return None, None, 0, None
        
        # Collect all directors
        director_ids = []
        director_names = []
        
        for member in crew:
            if member.get('job') == 'Director':
                dir_id = member.get('id')
                dir_name = member.get('name')
                if dir_id and dir_name:
                    director_ids.append(dir_id)
                    director_names.append(dir_name)
        
        if not director_ids:
            return None, None, 0, None
        
        # Primary director is first
        primary_dir_id = director_ids[0]
        primary_dir_name = director_names[0]
        num_directors = len(director_ids)
        other_directors = '|'.join(director_names[1:]) if len(director_names) > 1 else None
        
        return primary_dir_id, primary_dir_name, num_directors, other_directors

    def get_director_full_profile(self, director_id, target_release_date_id, target_collection_id):
    
        # Check cache first (thread-safe with double-check pattern)
        with self._cache_lock:
            if director_id in self.director_cache:
                credits_data = self.director_cache[director_id]
            else:
                credits_data = None
    
        if credits_data is None:
            url = f"{TMDB_BASE_URL}/person/{director_id}/movie_credits"
            resp = blocking_request(url, params={'api_key': TMDB_API_KEY}, session_obj=session)
            if not resp: 
                return None
            try:
                credits_data = resp.json().get('crew', [])
                with self._cache_lock:
                    self.director_cache[director_id] = credits_data
            except:
                return None
    
        # Parse target date
        try:
            target_date_str = str(int(target_release_date_id))
            target_date = pd.to_datetime(target_date_str, format='%Y%m%d')
        except:
            return None
    
        # Filter directed movies only (before target date)
        directed_movies = []
        genre_counter = Counter()
    
        for movie in credits_data:
            if movie.get('job') != 'Director':
                continue
        
            r_date_str = movie.get('release_date', '')
            if not r_date_str:
                continue
        
            try:
                r_date = pd.to_datetime(r_date_str)
                if r_date.date() >= target_date.date():
                    continue
            
                # Collect genre IDs
                genre_ids = movie.get('genre_ids', [])
                for gid in genre_ids:
                    genre_counter[gid] += 1
            
                directed_movies.append({
                    'id': movie.get('id'),
                    'title': movie.get('title', ''),
                    'release_date': r_date
                })
            except:
                continue
    
        if not directed_movies:
            return None
    
        # Sort by date (newest first)
        directed_movies.sort(key=lambda x: x['release_date'], reverse=True)
    
        # Collect budget, runtime, box office, and franchise overlap
        valid_runtimes = []
        valid_budgets_raw = []
        valid_budgets_adj = []
        franchise_films_count = 0
        short_films_count = 0
        films_with_budget = 0
    
        # Batch check cache for movie details
        uncached_movie_ids = []
        with self._cache_lock:
            for movie in directed_movies:
                if movie['id'] not in self.movie_details_cache:
                    uncached_movie_ids.append(movie['id'])
    
        # Fetch uncached movie details
        for movie_id in uncached_movie_ids:
            details_url = f"{TMDB_BASE_URL}/movie/{movie_id}"
            details_resp = blocking_request(details_url, params={'api_key': TMDB_API_KEY}, session_obj=session)
        
            if details_resp:
                try:
                    details = details_resp.json()
                    with self._cache_lock:
                        self.movie_details_cache[movie_id] = details
                except:
                    with self._cache_lock:
                        self.movie_details_cache[movie_id] = {}
    
        # Process all movies using cached details
        for movie in directed_movies:
            movie_id = movie['id']
        
            with self._cache_lock:
                details = self.movie_details_cache.get(movie_id, {})
        
            # Runtime
            runtime = details.get('runtime', 0)
            if runtime and runtime > 0:
                valid_runtimes.append(runtime)
                if runtime < 45:
                    short_films_count += 1
        
            # Budget (TMDb)
            tmdb_budget = details.get('budget', 0)
            if tmdb_budget and isinstance(tmdb_budget, (int, float)) and tmdb_budget > 10000:
                budget = int(tmdb_budget)
                if 10000 < budget < 1000000000:
                    films_with_budget += 1
                    valid_budgets_raw.append(int(budget))
                    budget_adj = adjust_for_inflation(budget, movie['release_date'], INFLATION_LOOKUP)
                    valid_budgets_adj.append(int(budget_adj) if budget_adj else int(budget))
        
            # Franchise overlap check
            if target_collection_id and target_collection_id > 0:
                belongs_to_collection = details.get('belongs_to_collection')
                if belongs_to_collection:
                    movie_collection_id = belongs_to_collection.get('id')
                    if movie_collection_id == target_collection_id:
                        franchise_films_count += 1
    
        # Calculate averages and medians
        avg_runtime_all = sum(valid_runtimes) / len(valid_runtimes) if valid_runtimes else 0
        director_avg_prior_budget_raw = sum(valid_budgets_raw) / len(valid_budgets_raw) if valid_budgets_raw else 0
        director_avg_prior_budget_adj = sum(valid_budgets_adj) / len(valid_budgets_adj) if valid_budgets_adj else 0
    
        # Budget coverage
        films_without_budget = len(directed_movies) - films_with_budget
        budget_coverage = round(films_with_budget / len(directed_movies) * 100, 1) if directed_movies else 0
    
        # Top 3 genres
        top_genres = []
        for genre_id, count in genre_counter.most_common(3):
            genre_name = GENRE_MAP.get(genre_id, f"Genre{genre_id}")
            top_genres.append(f"{genre_name}({count})")
    
        # Years since last film
        years_since_last_film = round((target_date - directed_movies[0]['release_date']).days / 365.25, 2) if directed_movies else 0
    
        return {
            'total_films': len(directed_movies),
            'director_prior_short_films': short_films_count,
            'director_prior_proper_films': len(directed_movies) - short_films_count,
            'years_active': round((target_date - directed_movies[-1]['release_date']).days / 365.25, 1) if directed_movies else 0,
            'years_since_last_film': years_since_last_film,
            'avg_runtime_L10': round(avg_runtime_all, 1),
            'director_avg_prior_budget_raw': int(director_avg_prior_budget_raw),
            'director_avg_prior_budget_adj': int(director_avg_prior_budget_adj),
            'director_films_with_budget': films_with_budget,
            'director_films_missing_budget': films_without_budget,
            'director_budget_coverage_pct': budget_coverage,
            'director_prior_franchise_films': franchise_films_count,
            'top_genres': '|'.join(top_genres) if top_genres else None,
        }

    def _process_single_movie(self, row_data):
        movie_id, date_id, collection_id = row_data
        
        # Find Director
        dir_id, dir_name, num_directors, other_directors = self.get_director_for_movie(movie_id)
        if not dir_id:
            return None, f"Movie {movie_id}: ⚠️ No Director Found"
        
        # Get profile (point-in-time, only prior films)
        profile = self.get_director_full_profile(dir_id, date_id, collection_id)
        
        # Build row
        if not profile:
            # First-time director - all zeros
            result = {
                'movie_id': movie_id,
                'num_directors': num_directors,
                'director_id': dir_id,
                'director_name': dir_name,
                'other_directors': other_directors,
                'total_films_before': 0,
                'director_prior_short_films': 0,
                'director_prior_proper_films': 0,
                'years_active': 0,
                'years_since_last_film': 0,
                'avg_runtime_L10': 0,
                'director_avg_prior_budget_raw': 0,
                'director_avg_prior_budget_adj': 0,
                'director_films_with_budget': 0,
                'director_films_missing_budget': 0,
                'director_budget_coverage_pct': 0,
                'director_prior_franchise_films': 0,
                'top_3_genres': None,
            }
            num_dir_str = f", {num_directors} dir{'s' if num_directors > 1 else ''}" if num_directors > 1 else ""
            return result, f"Movie {movie_id}: ⚠️ {dir_name} (First-time{num_dir_str})"
        else:
            # Has prior history
            result = {
                'movie_id': movie_id,
                'num_directors': num_directors,
                'director_id': dir_id,
                'director_name': dir_name,
                'other_directors': other_directors,
                'total_films_before': profile['total_films'],
                'director_prior_short_films': profile['director_prior_short_films'],
                'director_prior_proper_films': profile['director_prior_proper_films'],
                'years_active': profile['years_active'],
                'years_since_last_film': profile['years_since_last_film'],
                'avg_runtime_L10': profile['avg_runtime_L10'],
                'director_avg_prior_budget_raw': profile['director_avg_prior_budget_raw'],
                'director_avg_prior_budget_adj': profile['director_avg_prior_budget_adj'],
                'director_films_with_budget': profile['director_films_with_budget'],
                'director_films_missing_budget': profile['director_films_missing_budget'],
                'director_budget_coverage_pct': profile['director_budget_coverage_pct'],
                'director_prior_franchise_films': profile['director_prior_franchise_films'],
                'top_3_genres': profile['top_genres'],
            }
            budget_str = (
                f"${profile['director_avg_prior_budget_adj']/1e6:.0f}M avg "
                f"({profile['director_budget_coverage_pct']:.0f}% coverage)"
                if profile['director_avg_prior_budget_adj'] > 0
                else f"no budget ({profile['director_films_missing_budget']} films missing)"
            )
            franchise_str = (
                f", {profile['director_prior_franchise_films']} in franchise"
                if profile['director_prior_franchise_films'] > 0
                else ""
            )
            return (
                result,
                f"Movie {movie_id}: ✅ {dir_name} "
                f"({profile['total_films']} films, {budget_str}{franchise_str})"
            )

    def build_dataset(self):
        if not os.path.exists(INPUT_FILE):
            print(f"❌ {INPUT_FILE} not found. Run Fact Table Generator first.")
            return

        # Load checkpoint
        existing_df, processed_ids = load_checkpoint(OUTPUT_FILE, id_col='movie_id')
        
        fact_df = pd.read_csv(INPUT_FILE)
        
        print(f"🎬 Processing Director History for {len(fact_df)} movies (PARALLEL with {self.max_workers} workers)...")
        if processed_ids:
            print(f"   ⏭️ Skipping {len(processed_ids)} already processed movies")

        # Prepare items for parallel processing
        items_to_process = []
        for idx, row in fact_df.iterrows():
            movie_id = int(row['movie_id'])
            date_id = int(row['date_id'])
            collection_id = row.get('collection_id', 0)
            if pd.isna(collection_id):
                collection_id = 0
            else:
                collection_id = int(collection_id)
            
            # Skip invalid dates
            if date_id == 99991231:
                continue
            
            # Skip if already processed
            if is_already_processed(movie_id, processed_ids):
                continue
            
            items_to_process.append((movie_id, date_id, collection_id))
        
        if not items_to_process:
            print("✅ All movies already processed!")
            return
        
        print(f"   📋 {len(items_to_process)} movies to process")
        
        # Run parallel processing
        results = parallel_process(
            items=items_to_process,
            process_func=self._process_single_movie,
            max_workers=self.max_workers,
            desc="Director History"
        )
        
        # Collect successful results
        output_rows = []
        success_count = 0
        skip_count = 0
        
        for result, message in results:
            if result is not None:
                output_rows.append(result)
                success_count += 1
            else:
                skip_count += 1
        
        # Save all results
        if output_rows:
            existing_df = save_checkpoint(output_rows, existing_df, OUTPUT_FILE)
        
        print(f"\n🎉 Completed! {OUTPUT_FILE} saved")
        print(f"   Total: {len(existing_df)} directors")
        print(f"   This run: {success_count} processed, {skip_count} skipped (no director)")
        
        # Show summary
        if not existing_df.empty:
            print("\nSample:")
            display_cols = [
                'director_name',
                'total_films_before',
            ]
            display_cols = [c for c in display_cols if c in existing_df.columns]
            print(existing_df[display_cols].head(10))


if __name__ == "__main__":
    builder = DirectorHistoryBuilder(max_workers=5)
    builder.build_dataset()

In [22]:
# ==========================================
# CONFIGURATION
# ==========================================
INPUT_FILE = "Movie_Dimension.csv"
OUTPUT_FILE = "Dim_Cast.csv"

In [ ]:
# ==========================================
# CAST HISTORY BUILDER (PARALLEL)
# ==========================================
class CastHistoryBuilder:
    def __init__(self, max_workers=20):
        self.actor_cache = {}  # Cache actor credits
        self.director_cache = {}  # Cache director info
        self.costar_cache = {}  # Cache movie casts for co-star analysis
        self.movie_details_cache = {}  # Cache movie details for gross fetching
        self.max_workers = max_workers
        self._cache_lock = Lock()  # Thread-safe cache access
        print(f"✓ Cast builder initialized with {max_workers} parallel workers (TMDb only - USD budgets)")

    def get_credits_for_movie(self, movie_id):
        cache_key = f"credits_{movie_id}"
        
        with self._cache_lock:
            if cache_key in self.costar_cache:
                return self.costar_cache[cache_key]
        
        url = f"{TMDB_BASE_URL}/movie/{movie_id}/credits"
        resp = blocking_request(url, params={'api_key': TMDB_API_KEY}, session_obj=session)
        if not resp:
            return None
        
        try:
            credits = resp.json()
            with self._cache_lock:
                self.costar_cache[cache_key] = credits
            return credits
        except:
            return None

    def get_top_cast_for_movie(self, movie_id, limit=5):
        # Check if already cached (thread-safe)
        cache_key = f"{movie_id}_{limit}"
        with self._cache_lock:
            if cache_key in self.costar_cache:
                return self.costar_cache[cache_key]
        
        credits = self.get_credits_for_movie(movie_id)
        if not credits:
            return []
        
        cast = credits.get('cast', [])[:limit]
        with self._cache_lock:
            self.costar_cache[cache_key] = cast
        return cast
    
    def get_director_for_movie(self, movie_id):
        with self._cache_lock:
            if movie_id in self.director_cache:
                return self.director_cache[movie_id]
        
        credits = self.get_credits_for_movie(movie_id)
        if not credits:
            return None
        
        try:
            crew = credits.get('crew', [])
            for member in crew:
                if member.get('job') == 'Director':
                    director_id = member.get('id')
                    with self._cache_lock:
                        self.director_cache[movie_id] = director_id
                    return director_id
        except:
            pass
        return None

    def get_actor_profile(self, person_id, target_date, target_collection_id, target_genres):
        # Check cache (thread-safe)
        with self._cache_lock:
            if person_id in self.actor_cache:
                credits_data = self.actor_cache[person_id]
            else:
                credits_data = None
        
        if credits_data is None:
            url = f"{TMDB_BASE_URL}/person/{person_id}/movie_credits"
            resp = blocking_request(url, params={'api_key': TMDB_API_KEY}, session_obj=session)
            if not resp: 
                return None
            try:
                credits_data = resp.json().get('cast', [])
                with self._cache_lock:
                    self.actor_cache[person_id] = credits_data
            except:
                return None
        
        # Filter movies before target date (point-in-time)
        acted_movies = []
        genre_counter = Counter()
        
        for movie in credits_data:
            r_date_str = movie.get('release_date', '')
            if not r_date_str:
                continue
            
            try:
                r_date = pd.to_datetime(r_date_str)
                if r_date.date() >= target_date.date():  # Leakage prevention
                    continue
                
                # Collect genre IDs
                genre_ids = movie.get('genre_ids', [])
                for gid in genre_ids:
                    genre_counter[gid] += 1
                
                acted_movies.append({
                    'id': movie.get('id'),
                    'title': movie.get('title', ''),
                    'release_date': r_date,
                    'genre_ids': genre_ids
                })
            except:
                continue
        
        if not acted_movies:
            return None
        
        # Sort by date (newest first)
        acted_movies.sort(key=lambda x: x['release_date'], reverse=True)
        
        # Get budgets from latest 10 movies (point-in-time)
        latest_10 = acted_movies[:10]
        valid_budgets = []
        has_franchise_appearance = False
        
        # === STEP 1: Check ALL actor movies for franchise appearance ===
        
        # Get ALL movie IDs for franchise checking 
        all_movie_ids = [m['id'] for m in acted_movies]
        
        # Batch fetch uncached movie details for latest 10 for budget calculation
        with self._cache_lock:
            uncached_ids_budget = [m['id'] for m in latest_10 if m['id'] not in self.movie_details_cache]
        
        for movie_id in uncached_ids_budget:
            details_url = f"{TMDB_BASE_URL}/movie/{movie_id}"
            details_resp = blocking_request(details_url, params={'api_key': TMDB_API_KEY}, session_obj=session)
            if details_resp:
                try:
                    with self._cache_lock:
                        self.movie_details_cache[movie_id] = details_resp.json()
                except:
                    with self._cache_lock:
                        self.movie_details_cache[movie_id] = {}
        
        # Process latest 10 for budget calculation
        for movie in latest_10:
            movie_id = movie['id']
            with self._cache_lock:
                details = self.movie_details_cache.get(movie_id, {})
            
            # Budget (TMDb)
            budget = details.get('budget', 0)
            # Validate: must be numeric, positive, and reasonable (>$10K, <$1B)
            if budget and isinstance(budget, (int, float)) and 10000 < budget < 1000000000:
                # Adjust budget for inflation based on movie's release date
                budget_adj = adjust_for_inflation(budget, movie['release_date'], INFLATION_LOOKUP)
                valid_budgets.append(int(budget_adj) if budget_adj else int(budget))
        
        # === STEP 2: Check franchise appearance across ALL prior movies ===
        if target_collection_id and target_collection_id > 0:
            # First check the movies already cached
            with self._cache_lock:
                cached_details = {mid: self.movie_details_cache.get(mid) for mid in all_movie_ids 
                                 if mid in self.movie_details_cache}
            
            for movie_id, details in cached_details.items():
                if details:
                    belongs_to_collection = details.get('belongs_to_collection')
                    if belongs_to_collection:
                        movie_collection_id = belongs_to_collection.get('id')
                        if movie_collection_id == target_collection_id:
                            has_franchise_appearance = True
                            break
            
            # Fetch details for uncached movies 
            if not has_franchise_appearance:
                with self._cache_lock:
                    uncached_for_franchise = [mid for mid in all_movie_ids if mid not in self.movie_details_cache]
                
                for movie_id in uncached_for_franchise:
                    details_url = f"{TMDB_BASE_URL}/movie/{movie_id}"
                    details_resp = blocking_request(details_url, params={'api_key': TMDB_API_KEY}, session_obj=session)
                    if details_resp:
                        try:
                            details = details_resp.json()
                            with self._cache_lock:
                                self.movie_details_cache[movie_id] = details
                            
                            belongs_to_collection = details.get('belongs_to_collection')
                            if belongs_to_collection:
                                movie_collection_id = belongs_to_collection.get('id')
                                if movie_collection_id == target_collection_id:
                                    has_franchise_appearance = True
                                    break  # Found a match, no need to continue
                        except:
                            with self._cache_lock:
                                self.movie_details_cache[movie_id] = {}
        
        avg_budget_L10 = sum(valid_budgets) / len(valid_budgets) if valid_budgets else 0
        
        # Calculate genre fit: how many of actor's genres match target movie's genres
        genre_fit_count = 0
        if target_genres:
            actor_genres = set(genre_counter.keys())
            genre_fit_count = len(actor_genres & set(target_genres))
        
        # Years active
        if acted_movies:
            first_date = acted_movies[-1]['release_date']
            years_active = (target_date - first_date).days / 365.25
        else:
            years_active = 0
        
        # Top 3 genres
        top_genres = [GENRE_MAP.get(gid, f"Genre{gid}") for gid, _ in genre_counter.most_common(3)]
        
        # Return movie IDs for co-star and director analysis
        movie_ids = [m['id'] for m in acted_movies]
        
        return {
            'total_films': len(acted_movies),
            'avg_budget_L10': int(avg_budget_L10),
            'has_franchise_appearance': has_franchise_appearance,
            'genre_fit_count': genre_fit_count,
            'years_active': round(years_active, 1),
            'top_genres': '|'.join(top_genres) if top_genres else None,
            'movie_ids': movie_ids
        }
    
    def calculate_costar_affinity(self, cast_profiles):
        
        actor_ids = list(cast_profiles.keys())
        collaboration_counts = []
        
        for i, actor_id in enumerate(actor_ids):
            if not cast_profiles[actor_id]:
                continue
            
            actor_movies = set(cast_profiles[actor_id]['movie_ids'])
            collaborations = 0
            
            # Check overlap with other actors
            for j, other_id in enumerate(actor_ids):
                if i == j or not cast_profiles[other_id]:
                    continue
                
                other_movies = set(cast_profiles[other_id]['movie_ids'])
                overlap = len(actor_movies & other_movies)
                if overlap > 0:
                    collaborations += overlap
            
            if collaborations > 0:
                collaboration_counts.append(collaborations)
        
        # Average across all 5 actors (including zeros)
        total_collabs = sum(collaboration_counts)
        return total_collabs / 5 if total_collabs > 0 else 0
    
    def calculate_director_collaborations(self, cast_profiles, director_id):
        
        if not director_id:
            return 0, 0, 0
        
        # Check cache for director's filmography 
        filmography_key = f"filmography_{director_id}"
        with self._cache_lock:
            if filmography_key in self.director_cache:
                director_movies = self.director_cache[filmography_key]
            else:
                director_movies = None
        
        if director_movies is None:
            # Get director's filmography
            url = f"{TMDB_BASE_URL}/person/{director_id}/movie_credits"
            resp = blocking_request(url, params={'api_key': TMDB_API_KEY}, session_obj=session)
            if not resp:
                return 0, 0, 0
            
            try:
                director_movies = set([m['id'] for m in resp.json().get('crew', []) if m.get('job') == 'Director'])
                # Cache the movie set 
                with self._cache_lock:
                    self.director_cache[f"filmography_{director_id}"] = director_movies
            except:
                return 0, 0, 0
        
        actors_with_collab = 0
        total_collab_films = 0
        
        for actor_id, profile in cast_profiles.items():
            if not profile:
                continue
            
            actor_movies = set(profile['movie_ids'])
            overlap = len(actor_movies & director_movies)
            
            if overlap > 0:
                actors_with_collab += 1
                total_collab_films += overlap
        
        # Average across all 5 actors (not just those with collaborations)
        avg_collab_films = total_collab_films / 5
        
        return actors_with_collab, round(avg_collab_films, 2), total_collab_films
    
    def get_crew_metrics(self, movie_id):

        credits = self.get_credits_for_movie(movie_id)
        if not credits:
            return 0
        
        try:
            crew = credits.get('crew', [])
            # Standard TMDb department name is 'Visual Effects'
            vfx_count = sum(1 for member in crew if member.get('department') == 'Visual Effects')
            return vfx_count
        except:
            return 0

    def _process_single_movie(self, row_data):
        movie_id, date_id, collection_id, genres_str = row_data
        
        # Parse target date
        try:
            target_date = pd.to_datetime(str(date_id), format='%Y%m%d')
        except:
            return None, f"Movie {movie_id}: ❌ Invalid date"
        
        # Get target movie genres (reverse lookup)
        target_genres = []
        if genres_str and not pd.isna(genres_str):
            for g in str(genres_str).split('|'):
                for gid, gname in GENRE_MAP.items():
                    if gname == g:
                        target_genres.append(gid)
                        break
        
        # Get full cast 
        full_cast = self.get_top_cast_for_movie(movie_id, limit=9999)
        total_cast_size = len(full_cast) if full_cast else 0
        top_cast = full_cast[:5] if full_cast else []
        
        if not top_cast:
            # No cast - single row with zeros
            result = {
                'movie_id': movie_id,
                'date_id': date_id,
                'total_cast_size': total_cast_size,
                'avg_costar_affinity': 0,
                'actors_worked_with_director': 0,
                'avg_films_with_director': 0,
                'total_films_with_director': 0,
                'avg_years_active': 0,
                'top_3_genres': None,
                'avg_films_per_actor': 0,
                'avg_budget_L10': 0,
                'returning_franchise_actors': 0,
                'avg_genre_fit': 0,
                'vfx_crew_count': 0
            }
            return result, f"Movie {movie_id}: ⚠️ No Cast"
        
        # Get director
        director_id = self.get_director_for_movie(movie_id)
        
        # Get profiles for all 5 actors
        cast_profiles = {}
        genre_counter = Counter()
        
        for actor in top_cast:
            person_id = actor.get('id')
            profile = self.get_actor_profile(person_id, target_date, collection_id, target_genres)
            cast_profiles[person_id] = profile
            
            # Aggregate genres
            if profile and profile['top_genres']:
                for genre in profile['top_genres'].split('|'):
                    genre_counter[genre] += 1
        
        # Calculate metrics
        costar_affinity = self.calculate_costar_affinity(cast_profiles)
        actors_with_dir, avg_films_with_dir, total_films_with_dir = self.calculate_director_collaborations(cast_profiles, director_id)
        
        # Aggregate actor stats, divide by 5 to include actors with no history
        valid_profiles = [p for p in cast_profiles.values() if p]
        
        # Sum totals and divide by 5 (all actors including those with 0 films)
        total_years_active = sum([p['years_active'] for p in valid_profiles])
        total_films = sum([p['total_films'] for p in valid_profiles])
        total_budgets = sum([p['avg_budget_L10'] for p in valid_profiles])
        returning_franchise_actors = sum([1 for p in valid_profiles if p['has_franchise_appearance']])
        total_genre_fit = sum([p['genre_fit_count'] for p in valid_profiles])
        
        avg_years_active = total_years_active / 5
        avg_films_per_actor = total_films / 5
        avg_budget_L10 = total_budgets / 5
        avg_genre_fit = total_genre_fit / 5
        
        # Get VFX Crew Count
        vfx_crew_count = self.get_crew_metrics(movie_id)

        # Top 3 genres across all cast
        top_3_genres = '|'.join([g for g, _ in genre_counter.most_common(3)]) if genre_counter else None
        
        # Build single row for this movie
        result = {
            'movie_id': movie_id,
            'date_id': date_id,
            'total_cast_size': total_cast_size,
            'avg_costar_affinity': round(costar_affinity, 2),
            'actors_worked_with_director': actors_with_dir,
            'avg_films_with_director': avg_films_with_dir,
            'total_films_with_director': total_films_with_dir,
            'avg_years_active': round(avg_years_active, 1),
            'top_3_genres': top_3_genres,
            'avg_films_per_actor': round(avg_films_per_actor, 1),
            'avg_budget_L10': int(avg_budget_L10),
            'returning_franchise_actors': returning_franchise_actors,
            'avg_genre_fit': round(avg_genre_fit, 2),
            'vfx_crew_count': vfx_crew_count
        }
        
        return result, f"Movie {movie_id}: ✅ Cast: {total_cast_size} | ${avg_budget_L10/1e6:.0f}M | Franchise: {returning_franchise_actors}/5"

    def build_dataset(self):
        if not os.path.exists(INPUT_FILE):
            print(f"❌ {INPUT_FILE} not found. Run Movie Dimension Generator first.")
            return

        # Load checkpoint
        existing_df, processed_ids = load_checkpoint(OUTPUT_FILE, id_col='movie_id')
        
        fact_df = pd.read_csv(INPUT_FILE)
        
        print(f"🎬 Processing Cast Metrics for {len(fact_df)} movies (PARALLEL with {self.max_workers} workers)")
        print(f"   Features: cast_size, budget_L10, returning_actors, genre_fit, director_collabs")
        if processed_ids:
            print(f"   ⏭️ Skipping {len(processed_ids)} already processed movies")

        # Prepare items for parallel processing
        items_to_process = []
        for idx, row in fact_df.iterrows():
            movie_id = int(row['movie_id'])
            date_id = int(row['date_id'])
            
            if date_id == 99991231:
                continue
            
            # Skip if already processed
            if is_already_processed(movie_id, processed_ids):
                continue
            
            # Get collection_id and genres for this row
            collection_id = row.get('collection_id', 0)
            if pd.isna(collection_id):
                collection_id = 0
            else:
                collection_id = int(collection_id)
            
            genres_str = row.get('genres', '')
            
            items_to_process.append((movie_id, date_id, collection_id, genres_str))
        
        if not items_to_process:
            print("✅ All movies already processed!")
            return
        
        print(f"   📋 {len(items_to_process)} movies to process")
        
        # Run parallel processing
        results = parallel_process(
            items=items_to_process,
            process_func=self._process_single_movie,
            max_workers=self.max_workers,
            desc="Cast History"
        )
        
        # Collect successful results
        output_rows = []
        success_count = 0
        
        for result, message in results:
            if result is not None:
                output_rows.append(result)
                success_count += 1
        
        # Save all results
        if output_rows:
            existing_df = save_checkpoint(output_rows, existing_df, OUTPUT_FILE)
        
        print(f"\n🎉 Completed! {OUTPUT_FILE} saved")
        print(f"   Total: {len(existing_df)} movie cast metrics")
        print(f"   This run: {success_count} processed")
        
        # Show summary
        if not existing_df.empty:
            print("\nSample:")
            # Display available columns (handle old checkpoint files with different column names)
            display_cols = ['movie_id']
            for col in ['total_cast_size', 'avg_budget_L10', 'returning_franchise_actors', 'avg_genre_fit', 'total_films_with_director']:
                if col in existing_df.columns:
                    display_cols.append(col)
            print(existing_df[display_cols].head())

# Run it
if __name__ == "__main__":
    builder = CastHistoryBuilder(max_workers=20)
    builder.build_dataset()

In [ ]:
#==========================================
# CONFIGURATION
# ==========================================
INPUT_FILE = "Movie_Dimension.csv"
OUTPUT_FILE = "Dim_Production_Company.csv"

# Comprehensive mapping of production companies to their parent studio group
STUDIO_GROUP_MAPPING = {
    # ===== DISNEY GROUP =====
    "Walt Disney Pictures": "Disney",
    "Walt Disney Productions": "Disney",
    "Walt Disney Animation Studios": "Disney",
    "Disneytoon Studios": "Disney",
    "Touchstone Pictures": "Disney",
    "Hollywood Pictures": "Disney",
    # Pixar (Disney subsidiary)
    "Pixar": "Disney",
    "Pixar Animation Studios": "Disney",
    # Marvel (Disney subsidiary)
    "Marvel Studios": "Disney",
    "Marvel Films": "Disney",
    "Marvel Entertainment": "Disney",
    # Lucasfilm (Disney subsidiary)
    "Lucasfilm": "Disney",
    "Lucasfilm Ltd.": "Disney",
    # Fox (acquired by Disney 2019)
    "20th Century Fox": "Disney",
    "Twentieth Century Fox": "Disney",
    "Twentieth Century Fox Film Corporation": "Disney",
    "20th Century Studios": "Disney",
    "Fox 2000 Pictures": "Disney",
    "Fox Searchlight Pictures": "Disney",
    "Searchlight Pictures": "Disney",
    # ===== WARNER BROS GROUP =====
    "Warner Bros.": "Warner Bros",
    "Warner Bros. Pictures": "Warner Bros",
    "Warner Bros. Animation": "Warner Bros",
    "Warner Animation Group": "Warner Bros",
    "New Line Cinema": "Warner Bros",
    "DC Films": "Warner Bros",
    "DC Entertainment": "Warner Bros",
    "DC Studios": "Warner Bros",
    "Castle Rock Entertainment": "Warner Bros",
    # ===== UNIVERSAL GROUP =====
    "Universal Pictures": "Universal",
    "Universal Studios": "Universal",
    "Universal Pictures International": "Universal",
    "Universal Animation Studios": "Universal",
    "Focus Features": "Universal",
    "Working Title Films": "Universal",
    # DreamWorks Animation (NBCUniversal subsidiary)
    "DreamWorks Animation": "Universal",
    # Illumination (Universal subsidiary)
    "Illumination": "Universal",
    "Illumination Entertainment": "Universal",
    # ===== SONY/COLUMBIA GROUP =====
    "Columbia Pictures": "Sony",
    "Sony Pictures": "Sony",
    "Sony Pictures Entertainment": "Sony",
    "Sony Pictures Animation": "Sony",
    "TriStar Pictures": "Sony",
    "Screen Gems": "Sony",
    "Sony Pictures Classics": "Sony",
    "Columbia TriStar": "Sony",
    "Revolution Studios": "Sony",
    # ===== PARAMOUNT GROUP =====
    "Paramount Pictures": "Paramount",
    "Paramount Animation": "Paramount",
    "Nickelodeon Movies": "Paramount",
    "Paramount Players": "Paramount",
    "Paramount Vantage": "Paramount",
    "Republic Pictures": "Paramount",
    # ===== LIONSGATE GROUP =====
    "Lionsgate": "Lionsgate",
    "Lionsgate Films": "Lionsgate",
    "Lions Gate Films": "Lionsgate",
    "Summit Entertainment": "Lionsgate",
    "Mandate Pictures": "Lionsgate",
    "Pantelion Films": "Lionsgate",
    # ===== MGM GROUP =====
    "Metro-Goldwyn-Mayer": "MGM",
    "Metro-Goldwyn-Mayer (MGM)": "MGM",
    "MGM": "MGM",
    "United Artists": "MGM",
    "United Artists Releasing": "MGM",
    "Orion Pictures": "MGM",
    # ===== DREAMWORKS (Independent/Various) =====
    "DreamWorks Pictures": "DreamWorks",
    "DreamWorks SKG": "DreamWorks",
    "DreamWorks": "DreamWorks",
    # ===== AMBLIN =====
    "Amblin Entertainment": "Amblin",
    "Amblin Partners": "Amblin",
    # ===== LEGENDARY =====
    "Legendary Pictures": "Legendary",
    "Legendary Entertainment": "Legendary",
    # ===== BLUMHOUSE =====
    "Blumhouse Productions": "Blumhouse",
    "Blumhouse": "Blumhouse",
    # ===== A24 =====
    "A24": "A24",
    # ===== REGENCY =====
    "Regency Enterprises": "Regency",
    "New Regency Productions": "Regency",
    "New Regency": "Regency",
    # ===== VILLAGE ROADSHOW =====
    "Village Roadshow Pictures": "Village Roadshow",
    # ===== SKYDANCE =====
    "Skydance Media": "Skydance",
    "Skydance Productions": "Skydance",
    "Skydance Animation": "Skydance",
    # ===== STREAMING PLATFORMS =====
    "Netflix": "Netflix",
    "Netflix Studios": "Netflix",
    "Amazon Studios": "Amazon",
    "Amazon MGM Studios": "Amazon",
    "Prime Video": "Amazon",
    "Apple Studios": "Apple",
    "Apple TV+": "Apple",
    "Apple Original Films": "Apple",
    "HBO Films": "HBO",
    "HBO Max": "HBO",
    "Max Original Film": "HBO",
    "Hulu": "Hulu",
    "Disney+": "Disney",
    "Peacock": "Universal",
    # ===== OTHER US STUDIOS =====
    "Blue Sky Studios": "Blue Sky",
    "LAIKA": "LAIKA",
    "LAIKA Entertainment": "LAIKA",
    "Aardman Animations": "Aardman",
    "Miramax": "Miramax",
    "Miramax Films": "Miramax",
    "The Weinstein Company": "Weinstein",
    "Weinstein Company": "Weinstein",
    "Relativity Media": "Relativity",
    "STX Entertainment": "STX",
    "STXfilms": "STX",
    "Annapurna Pictures": "Annapurna",
    "NEON": "NEON",
    "FilmNation Entertainment": "FilmNation",
    "Participant": "Participant",
    "Participant Media": "Participant",
    "Entertainment One": "eOne",
    "eOne": "eOne",
    "Imagine Entertainment": "Imagine",
    "Plan B Entertainment": "Plan B",
    "Bad Robot Productions": "Bad Robot",
    "Bad Robot": "Bad Robot",
    "Monkeypaw Productions": "Monkeypaw",
    "Happy Madison Productions": "Happy Madison",
    "Platinum Dunes": "Platinum Dunes",
    "Original Film": "Original Film",
    "Atomic Monster": "Atomic Monster",
    # ========================================
    # JAPANESE STUDIOS
    # ========================================
    "Toho Co., Ltd.": "Toho",
    "Toho": "Toho",
    "Toho Company": "Toho",
    "Studio Ghibli": "Ghibli",
    "Toei Company": "Toei",
    "Toei Animation": "Toei",
    "Toei": "Toei",
    "Shochiku": "Shochiku",
    "Shochiku Co., Ltd.": "Shochiku",
    "Nikkatsu": "Nikkatsu",
    "Nikkatsu Corporation": "Nikkatsu",
    "Kadokawa": "Kadokawa",
    "Kadokawa Corporation": "Kadokawa",
    "Kadokawa Pictures": "Kadokawa",
    "Aniplex": "Aniplex",
    "CoMix Wave Films": "CoMix Wave",
    "Madhouse": "Madhouse",
    "Bones": "Bones",
    "Sunrise": "Sunrise",
    "Production I.G": "Production I.G",
    "MAPPA": "MAPPA",
    "Wit Studio": "Wit Studio",
    "ufotable": "ufotable",
    "Kyoto Animation": "Kyoto Animation",
    "A-1 Pictures": "A-1 Pictures",
    "CloverWorks": "CloverWorks",
    "TMS Entertainment": "TMS",
    "Shin-Ei Animation": "Shin-Ei",
    "OLM": "OLM",
    "Gainax": "Gainax",
    "Khara": "Khara",
    "Studio Khara": "Khara",
    # ========================================
    # CHINESE STUDIOS
    # ========================================
    "China Film Group Corporation": "China Film Group",
    "China Film Group": "China Film Group",
    "China Film Co., Ltd.": "China Film Group",
    "Huayi Brothers": "Huayi Brothers",
    "Huayi Brothers Media": "Huayi Brothers",
    "Huayi Brothers Pictures": "Huayi Brothers",
    "Bona Film Group": "Bona Film",
    "Bona Film": "Bona Film",
    "Beijing Enlight Pictures": "Enlight Pictures",
    "Enlight Pictures": "Enlight Pictures",
    "Enlight Media": "Enlight Pictures",
    "Beijing Jingxi Culture & Tourism Co., Ltd.": "Jingxi Culture",
    "Wanda Pictures": "Wanda",
    "Wanda Media": "Wanda",
    "Wanda Film": "Wanda",
    "Alibaba Pictures": "Alibaba Pictures",
    "Alibaba Pictures Group": "Alibaba Pictures",
    "Tencent Pictures": "Tencent Pictures",
    "iQIYI Pictures": "iQIYI",
    "iQIYI": "iQIYI",
    "Bilibili": "Bilibili",
    "Shanghai Film Group": "Shanghai Film",
    "Shanghai Film Studio": "Shanghai Film",
    "Beijing Culture": "Beijing Culture",
    "Light Chaser Animation": "Light Chaser",
    "Light Chaser Animation Studios": "Light Chaser",
    "Horgos Coloroom Pictures": "Coloroom Pictures",
    # ========================================
    # KOREAN STUDIOS
    # ========================================
    "CJ Entertainment": "CJ Entertainment",
    "CJ E&M": "CJ Entertainment",
    "CJ ENM": "CJ Entertainment",
    "CJ E&M Film Financing & Investment Entertainment & Comics": "CJ Entertainment",
    "Showbox": "Showbox",
    "Showbox/Mediaplex": "Showbox",
    "NEW": "Next Entertainment World",
    "Next Entertainment World": "Next Entertainment World",
    "Lotte Entertainment": "Lotte Entertainment",
    "Lotte Cultureworks": "Lotte Entertainment",
    "Barunson E&A": "Barunson",
    "Barunson": "Barunson",
    "Studio Dragon": "Studio Dragon",
    "Contents Panda": "Contents Panda",
    "Megabox Plus M": "Megabox",
    # ========================================
    # INDIAN STUDIOS 
    # ========================================
    "Yash Raj Films": "Yash Raj Films",
    "YRF": "Yash Raj Films",
    "Dharma Productions": "Dharma Productions",
    "Red Chillies Entertainment": "Red Chillies",
    "Eros International": "Eros",
    "Eros": "Eros",
    "UTV Motion Pictures": "UTV",
    "UTV": "UTV",
    "T-Series": "T-Series",
    "T-Series Films": "T-Series",
    "Zee Studios": "Zee Studios",
    "Reliance Entertainment": "Reliance Entertainment",
    "Excel Entertainment": "Excel Entertainment",
    "Phantom Films": "Phantom Films",
    "Viacom18 Studios": "Viacom18",
    "Viacom18 Motion Pictures": "Viacom18",
    "Fox Star Studios": "Disney",
    "Nadiadwala Grandson Entertainment": "Nadiadwala Grandson",
    "Maddock Films": "Maddock Films",
    "Colour Yellow Productions": "Colour Yellow",
    "AA Films": "AA Films",
    "PVR Pictures": "PVR",
    "Hombale Films": "Hombale Films",
    "Prasad Productions": "Prasad Productions",
    "Geetha Arts": "Geetha Arts",
    "Mythri Movie Makers": "Mythri Movie Makers",
    "DVV Entertainment": "DVV Entertainment",
    "UV Creations": "UV Creations",
    "Sri Venkateswara Creations": "Sri Venkateswara Creations",
    "Lyca Productions": "Lyca Productions",
    "Sun Pictures": "Sun Pictures",
    "2D Entertainment": "2D Entertainment",
    "Dream Warrior Pictures": "Dream Warrior",
    "Ayngaran International": "Ayngaran",
    # ========================================
    # BRITISH STUDIOS
    # ========================================
    "Working Title Films": "Universal",
    "Film4 Productions": "Film4",
    "Film4": "Film4",
    "BBC Films": "BBC Films",
    "BBC Film": "BBC Films",
    "StudioCanal": "StudioCanal",
    "StudioCanal UK": "StudioCanal",
    "Heyday Films": "Heyday Films",
    "DNA Films": "DNA Films",
    "Blueprint Pictures": "Blueprint Pictures",
    "Warp Films": "Warp Films",
    "See-Saw Films": "See-Saw Films",
    "British Film Institute": "BFI",
    "BFI": "BFI",
    "Pathé": "Pathé",
    "Pathé UK": "Pathé",
    "Eon Productions": "Eon",
    "Hammer Films": "Hammer",
    "Hammer Film Productions": "Hammer",
    "Ealing Studios": "Ealing",
    "Vertigo Films": "Vertigo Films",
    "Entertainment Film Distributors": "Entertainment Film",
    "Lionsgate UK": "Lionsgate",
    # ========================================
    # FRENCH STUDIOS
    # ========================================
    "Gaumont": "Gaumont",
    "Gaumont Film Company": "Gaumont",
    "Pathé": "Pathé",
    "Pathé Films": "Pathé",
    "EuropaCorp": "EuropaCorp",
    "Europa Corp": "EuropaCorp",
    "StudioCanal": "StudioCanal",
    "Canal+": "Canal+",
    "UGC": "UGC",
    "Wild Bunch": "Wild Bunch",
    "MK2 Films": "MK2",
    "MK2": "MK2",
    "Why Not Productions": "Why Not",
    "Les Films du Losange": "Films du Losange",
    "France 2 Cinéma": "France TV",
    "France 3 Cinéma": "France TV",
    "TF1 Films Production": "TF1",
    "M6 Films": "M6",
    "Orange Studio": "Orange Studio",
    "Paradis Films": "Paradis Films",
    # ========================================
    # GERMAN STUDIOS
    # ========================================
    "Constantin Film": "Constantin Film",
    "Constantin": "Constantin Film",
    "Bavaria Film": "Bavaria Film",
    "Bavaria": "Bavaria Film",
    "UFA": "UFA",
    "UFA Fiction": "UFA",
    "Studio Babelsberg": "Babelsberg",
    "Babelsberg": "Babelsberg",
    "X Filme Creative Pool": "X Filme",
    "Senator Film": "Senator Film",
    "Telepool": "Telepool",
    "Wiedemann & Berg Film": "Wiedemann & Berg",
    # ========================================
    # SPANISH/LATIN AMERICAN STUDIOS
    # ========================================
    "Atresmedia Cine": "Atresmedia",
    "Telecinco Cinema": "Telecinco",
    "MediaPro": "MediaPro",
    "The Mediapro Studio": "MediaPro",
    "MOD Producciones": "MOD",
    "Morena Films": "Morena Films",
    "El Deseo": "El Deseo",
    "Tornasol Films": "Tornasol",
    "Videocine": "Videocine",
    "Canana Films": "Canana",
    "Globo Filmes": "Globo",
    "O2 Filmes": "O2 Filmes",
    "Patagonik Film Group": "Patagonik",
    # ========================================
    # AUSTRALIAN/NEW ZEALAND STUDIOS
    # ========================================
    "Village Roadshow Pictures": "Village Roadshow",
    "Screen Australia": "Screen Australia",
    "Hopscotch Features": "Hopscotch",
    "See-Saw Films": "See-Saw Films",
    "Animal Logic": "Animal Logic",
    "WingNut Films": "WingNut Films",
    "New Zealand Film Commission": "NZFC",
    "Transmission Films": "Transmission",
    "Madman Entertainment": "Madman",
    # ========================================
    # SCANDINAVIAN STUDIOS
    # ========================================
    "Nordisk Film": "Nordisk Film",
    "SF Studios": "SF Studios",
    "Svensk Filmindustri": "SF Studios",
    "Zentropa": "Zentropa",
    "Zentropa Entertainment": "Zentropa",
    "Yellow Bird": "Yellow Bird",
    "Trust Nordisk": "Trust Nordisk",
    "Film i Väst": "Film i Väst",
    # ========================================
    # OTHER INTERNATIONAL STUDIOS
    # ========================================
    # Italian
    "Medusa Film": "Medusa",
    "01 Distribution": "01 Distribution",
    "Rai Cinema": "Rai Cinema",
    "Fandango": "Fandango",
    "Lucky Red": "Lucky Red",
    "Cattleya": "Cattleya",
    "Indigo Film": "Indigo Film",
    # Russian
    "Mosfilm": "Mosfilm",
    "CTB Film Company": "CTB",
    "Central Partnership": "Central Partnership",
    "Bazelevs": "Bazelevs",
    "Art Pictures Studio": "Art Pictures",
    # Middle Eastern
    "Rotana Studios": "Rotana",
    "Image Nation Abu Dhabi": "Image Nation",
    "MBC Studios": "MBC",
    "Saudi Film Council": "Saudi Film Council",
    # Southeast Asian
    "GDH 559": "GDH",
    "GTH": "GTH",
    "Sahamongkol Film International": "Sahamongkol",
    "Banjong Pisanthanakun": "Banjong",
    "Star Cinema": "Star Cinema",
    "Viva Films": "Viva Films",
    "Regal Films": "Regal Films",
    "PT Falcon": "Falcon Pictures",
    "Falcon Pictures": "Falcon Pictures",
    "Screenplay Films": "Screenplay Infiniti",
    "MD Pictures": "MD Pictures",
    # Hong Kong/Taiwan
    "Media Asia": "Media Asia",
    "Media Asia Entertainment": "Media Asia",
    "Emperor Motion Pictures": "Emperor",
    "Edko Films": "Edko",
    "China Star Entertainment": "China Star",
    "Milkyway Image": "Milkyway Image",
    "Universe Entertainment": "Universe Entertainment",
    "Central Motion Pictures": "Central Motion Pictures",
}

In [ ]:
# ==========================================
# PRODUCTION COMPANY FEATURE BUILDER
# ==========================================

# Publishers / distributors / non-production entities to exclude
NON_PRODUCTION_COMPANIES = {
    # Japanese Publishers (manga/light novel)
    "Shueisha", "Kodansha", "Shogakukan", "Kadokawa Shoten", "Hakusensha",
    "Square Enix", "ASCII Media Works", "Enterbrain", "Dengeki Bunko",
    "Ichijinsha", "Media Factory", "Houbunsha", "Akita Shoten",
    # Distributors (not producers)
    "A-1 Pictures Distribution", "Funimation", "Crunchyroll", "Sentai Filmworks",
    "VIZ Media", "Manga Entertainment", "Geneon", "ADV Films",
    # Music/Record Labels
    "Sony Music Entertainment Japan", "Lantis", "Flying Dog", "Pony Canyon",
    "King Records", "Avex Pictures", "NBC Universal Entertainment Japan",
    # TV Broadcasters (often listed but not lead producers)
    "TV Tokyo", "Nippon Television Network", "TBS", "Fuji Television Network",
    "MBS", "ABC Animation", "Yomiuri Telecasting Corporation",
    # Other non-production entities
    "Dentsu", "Toy's Factory", "Good Smile Company", "Bandai Namco Arts",
}

# Pre-computed lowercase set for efficient case-insensitive lookup
NON_PRODUCTION_COMPANIES_LOWER = {name.lower() for name in NON_PRODUCTION_COMPANIES}

# Studio inheritance - when a company rebrands, inherit filmography from predecessor
STUDIO_INHERITANCE = {
    "DC Studios": ["DC Films", "DC Entertainment", "DC Comics"],
    "Amazon MGM Studios": ["Amazon Studios", "MGM", "Metro-Goldwyn-Mayer"],
    "20th Century Studios": ["20th Century Fox", "Twentieth Century Fox", "Twentieth Century Fox Film Corporation"],
    "Searchlight Pictures": ["Fox Searchlight Pictures"],
    "Max Original Film": ["HBO Max", "HBO Films"],
    "Paramount+": ["CBS Films", "CBS Television Studios"],
}

def get_studio_group(company_name):
    if not company_name:
        return None
    # Exact match first 
    if company_name in STUDIO_GROUP_MAPPING:
        return STUDIO_GROUP_MAPPING[company_name]
    # Case-insensitive fallback
    name_lower = company_name.lower()
    for comp, group in STUDIO_GROUP_MAPPING.items():
        if comp.lower() == name_lower:
            return group
    return None

def is_major_studio(name):
    return get_studio_group(name) is not None

def is_actual_production_company(name):
    if not name:
        return False
    return name.lower() not in NON_PRODUCTION_COMPANIES_LOWER

def get_lead_company(companies_list, filmography_cache=None):
    if not companies_list:
        return None, None
    
    def get_film_count(comp_id):
        if filmography_cache and comp_id:
            return len(filmography_cache.get(comp_id, []))
        return 0
    
    # Filter to actual production companies (exclude publishers/distributors)
    actual_producers = [c for c in companies_list if is_actual_production_company(c.get('name', ''))]
    
    # If no actual producers found, use all companies
    if not actual_producers:
        actual_producers = companies_list
    
    # Find major studios among actual producers
    major_studios = [c for c in actual_producers if is_major_studio(c.get('name', ''))]
    
    if major_studios:
        if len(major_studios) == 1:
            return major_studios[0].get('name'), major_studios[0].get('id')
        # Pick major studio with most films
        best = max(major_studios, key=lambda c: get_film_count(c.get('id')))
        return best.get('name'), best.get('id')
    
    # Any actual production company with most films
    if filmography_cache and actual_producers:
        best = max(actual_producers, key=lambda c: get_film_count(c.get('id')))
        return best.get('name'), best.get('id')
    
    # First actual producer, or truly first company
    best_list = actual_producers if actual_producers else companies_list
    return best_list[0].get('name'), best_list[0].get('id')

# ==========================================
# PRODUCTION COMPANY BUILDER CLASS 
# ==========================================
class ProductionHistoryBuilder:
    def __init__(self, max_workers=20):
        self.company_filmography_cache = {}
        self.director_cache = {}
        self.director_filmography_cache = {}
        self.movie_details_cache = {}
        self.movie_credits_cache = {}
        self.collection_cache = {}
        self.company_search_cache = {}          # cache for /search/company
        self.max_workers = max_workers
        self._cache_lock = Lock()              # Thread-safe cache access
        
        # Track in-flight requests to avoid races / duplicate API calls
        self._pending_movie_details = set()
        self._pending_credits = set()
        self._pending_company_search = set()
        
        print(f"✓ Production Company builder initialized with {max_workers} parallel workers")

    # -------- Central movie-details cache --------
    def get_movie_details_cached(self, movie_id):
        # Check cache first
        with self._cache_lock:
            if movie_id in self.movie_details_cache:
                return self.movie_details_cache[movie_id]
            if movie_id in self._pending_movie_details:
                is_pending = True
            else:
                is_pending = False
                self._pending_movie_details.add(movie_id)
        
        # If another thread is fetching, wait briefly and reuse result if it arrives
        if is_pending:
            import time
            for _ in range(10):  # up to ~1 sec
                time.sleep(0.1)
                with self._cache_lock:
                    if movie_id in self.movie_details_cache:
                        return self.movie_details_cache[movie_id]
            # If still not available
            with self._cache_lock:
                self._pending_movie_details.add(movie_id)
        
        # Fetch outside the lock
        try:
            resp = blocking_request(
                f"{TMDB_BASE_URL}/movie/{movie_id}",
                params={"api_key": TMDB_API_KEY},
                session_obj=session
            )
            if resp:
                try:
                    data = resp.json()
                except Exception:
                    data = {}
            else:
                data = {}
            
            with self._cache_lock:
                self.movie_details_cache[movie_id] = data
            return data
        finally:
            with self._cache_lock:
                self._pending_movie_details.discard(movie_id)

    # -------- Simple wrapper around movie-details cache --------
    def get_production_companies(self, movie_id):
        details = self.get_movie_details_cached(movie_id)
        if not isinstance(details, dict):
            return []
        return details.get('production_companies', []) or []

    # -------- Credits / director logic --------
    def get_director_for_movie(self, movie_id):
        # Check caches
        with self._cache_lock:
            if movie_id in self.director_cache:
                return self.director_cache[movie_id]
            
            if movie_id in self.movie_credits_cache:
                crew = self.movie_credits_cache[movie_id]
            else:
                crew = None
                if movie_id in self._pending_credits:
                    is_pending = True
                else:
                    is_pending = False
                    self._pending_credits.add(movie_id)
        
        # If another thread is fetching credits, wait briefly
        if crew is None and is_pending:
            import time
            for _ in range(10):
                time.sleep(0.1)
                with self._cache_lock:
                    if movie_id in self.movie_credits_cache:
                        crew = self.movie_credits_cache[movie_id]
                        break
            if crew is None:
                with self._cache_lock:
                    self._pending_credits.add(movie_id)
        
        # Fetch credits if needed
        if crew is None:
            try:
                url = f"{TMDB_BASE_URL}/movie/{movie_id}/credits"
                resp = blocking_request(url, params={'api_key': TMDB_API_KEY}, session_obj=session)
                if not resp:
                    with self._cache_lock:
                        self.director_cache[movie_id] = None
                    return None
                try:
                    crew = resp.json().get('crew', []) or []
                    with self._cache_lock:
                        self.movie_credits_cache[movie_id] = crew
                except Exception:
                    with self._cache_lock:
                        self.director_cache[movie_id] = None
                    return None
            finally:
                with self._cache_lock:
                    self._pending_credits.discard(movie_id)
        
        # Extract director from crew
        for member in crew:
            if member.get('job') == 'Director':
                director_id = member.get('id')
                with self._cache_lock:
                    self.director_cache[movie_id] = director_id
                return director_id
        
        with self._cache_lock:
            self.director_cache[movie_id] = None
        return None
    
    def get_director_filmography(self, director_id):
        if not director_id:
            return set()
        
        with self._cache_lock:
            if director_id in self.director_filmography_cache:
                return self.director_filmography_cache[director_id]
        
        url = f"{TMDB_BASE_URL}/person/{director_id}/movie_credits"
        resp = blocking_request(url, params={'api_key': TMDB_API_KEY}, session_obj=session)
        if not resp:
            return set()
        try:
            movies = set(
                m['id'] for m in resp.json().get('crew', []) 
                if m.get('job') == 'Director'
            )
            with self._cache_lock:
                self.director_filmography_cache[director_id] = movies
            return movies
        except Exception:
            return set()

    # -------- Collection / franchise info --------
    def get_collection_info(self, collection_id, target_date):
        if not collection_id:
            return set(), {}
        
        with self._cache_lock:
            if collection_id in self.collection_cache:
                all_movies = self.collection_cache[collection_id]
            else:
                all_movies = None
        
        if all_movies is None:
            url = f"{TMDB_BASE_URL}/collection/{collection_id}"
            resp = blocking_request(url, params={'api_key': TMDB_API_KEY}, session_obj=session)
            if not resp:
                return set(), {}
            try:
                all_movies = resp.json().get('parts', []) or []
                with self._cache_lock:
                    self.collection_cache[collection_id] = all_movies
            except Exception:
                return set(), {}
        
        prior_ids = set()
        movie_producers = {}
        
        for m in all_movies:
            r_date_str = m.get('release_date', '')
            if not r_date_str:
                continue
            try:
                r_date = pd.to_datetime(r_date_str)
                if r_date.date() >= target_date.date():
                    continue
                movie_id = m.get('id')
                if not movie_id:
                    continue
                prior_ids.add(movie_id)
                prod_companies = self.get_production_companies(movie_id)
                company_ids = set(c.get('id') for c in prod_companies if c.get('id'))
                movie_producers[movie_id] = company_ids
            except Exception:
                continue
        
        return prior_ids, movie_producers

    # -------- Company filmography and search --------
    def get_company_filmography(self, company_id, target_date=None):
        with self._cache_lock:
            key = (company_id, target_date.date() if target_date else None)
            if key in self.company_filmography_cache:
                return self.company_filmography_cache[key]
            #if company_id in self.company_filmography_cache:
            #    return self.company_filmography_cache[company_id]
        
        all_movies = []
        page = 1
        max_pages = 30

        while page <= max_pages:
            url = f"{TMDB_BASE_URL}/discover/movie"
            params = {
                "api_key": TMDB_API_KEY,
                "with_companies": company_id,
                "sort_by": "primary_release_date.desc",
                "page": page,
            }
            if target_date is not None:
                params["primary_release_date.lte"] = target_date.strftime("%Y-%m-%d")

            resp = blocking_request(url, params=params, session_obj=session)
            if not resp:
                break
            try:
                data = resp.json()
                results = data.get("results", []) or []
                if not results:
                    break
                all_movies.extend(results)
                if page >= data.get("total_pages", 1):
                    break
                page += 1
            except Exception:
                break
        
        with self._cache_lock:
            self.company_filmography_cache[key] = all_movies
        return all_movies

    def _search_company_id(self, name):
        key = name.lower()
        with self._cache_lock:
            if key in self.company_search_cache:
                return self.company_search_cache[key]
            if key in self._pending_company_search:
                is_pending = True
            else:
                is_pending = False
                self._pending_company_search.add(key)
        
        if is_pending:
            import time
            for _ in range(10):
                time.sleep(0.1)
                with self._cache_lock:
                    if key in self.company_search_cache:
                        return self.company_search_cache[key]
            with self._cache_lock:
                self._pending_company_search.add(key)
        
        try:
            search_url = f"{TMDB_BASE_URL}/search/company"
            resp = blocking_request(
                search_url,
                params={'api_key': TMDB_API_KEY, 'query': name},
                session_obj=session
            )
            company_id = None
            if resp:
                try:
                    results = resp.json().get('results', []) or []
                    for result in results:
                        if result.get('name', '').lower() == key:
                            company_id = result.get('id')
                            break
                except Exception:
                    company_id = None
            with self._cache_lock:
                self.company_search_cache[key] = company_id
            return company_id
        finally:
            with self._cache_lock:
                self._pending_company_search.discard(key)

    # -------- Company profile --------
    def get_company_profile(self, company_id, company_name, target_date,
                            franchise_movie_producers, target_genres, current_director_id):
        all_movies = self.get_company_filmography(company_id, target_date)
        
        company_movies = []
        genre_counter = Counter()
        prior_films_in_genre = 0
        
        # Track movie IDs to prevent duplicates across base + inheritance 
        existing_ids = set()
        
        for movie in all_movies:
            r_date_str = movie.get("release_date", "")
            if not r_date_str:
                continue
            try:
                r_date = pd.to_datetime(r_date_str)
                if r_date.date() >= target_date.date():
                    continue
                
                mid = movie.get("id")
                if not mid or mid in existing_ids:
                    continue
                
                existing_ids.add(mid)
                genre_ids = movie.get("genre_ids", []) or []
                
                for gid in genre_ids:
                    genre_counter[gid] += 1
                
                if target_genres and any(gid in target_genres for gid in genre_ids):
                    prior_films_in_genre += 1
                
                company_movies.append({"id": mid, "release_date": r_date})
            except Exception:
                continue
        
        # STUDIO INHERITANCE: combine filmographies with predecessors
        used_inheritance = False
        if company_name in STUDIO_INHERITANCE:
            original_count = len(company_movies)
            for predecessor_name in STUDIO_INHERITANCE[company_name]:
                pred_id = self._search_company_id(predecessor_name)
                if not pred_id:
                    continue
                pred_movies = self.get_company_filmography(pred_id, target_date)
                for movie in pred_movies:
                    r_date_str = movie.get("release_date", "")
                    if not r_date_str:
                        continue
                    try:
                        r_date = pd.to_datetime(r_date_str)
                        if r_date.date() >= target_date.date():
                            continue
                        mid = movie.get("id")
                        if not mid or mid in existing_ids:
                            continue
                        existing_ids.add(mid)
                        genre_ids = movie.get("genre_ids", []) or []
                        for gid in genre_ids:
                            genre_counter[gid] += 1
                        if target_genres and any(gid in target_genres for gid in genre_ids):
                            prior_films_in_genre += 1
                        company_movies.append({"id": mid, "release_date": r_date})
                    except Exception:
                        continue
            if len(company_movies) > original_count:
                used_inheritance = True
                inherited_count = len(company_movies) - original_count
                print(f"  → Combined {inherited_count} predecessor films with {original_count} new films for {company_name} (total: {len(company_movies)})")
        
        if not company_movies:
            return {
                "num_prior_films": 0,
                "prior_films_in_genre": 0,
                "prior_franchise_films": 0,
                "is_major_studio": 1 if is_major_studio(company_name) else 0,
                "is_rebranded_studio": 0,
                "director_collab_count": 0,
                "top_3_genres": None,
                "avg_budget_L5": 0,
                "avg_movies_per_year": 0,
            }
        
        company_movies.sort(key=lambda x: x["release_date"], reverse=True)
        company_movie_ids = set(m["id"] for m in company_movies)
        
        # Franchise count
        prior_franchise_films = 0
        if franchise_movie_producers:
            for fmovie_id, producer_ids in franchise_movie_producers.items():
                if company_id in producer_ids:
                    prior_franchise_films += 1
        
        # Budget for latest 5 movies WITH VALID BUDGETS 
        MIN_VALID_BUDGET = 10_000           # $10K minimum 
        MAX_VALID_BUDGET = 600_000_000      # $600M maximum 
        
        valid_budgets = []
        movies_checked = 0
        max_movies_to_check = min(50, len(company_movies))  # scan up to 50 recent films
        
        for movie in company_movies:
            if len(valid_budgets) >= 5 or movies_checked >= max_movies_to_check:
                break
            movies_checked += 1
            
            mid = movie["id"]
            movie_details = self.get_movie_details_cached(mid)
            budget = movie_details.get("budget", 0)
            
            if isinstance(budget, (int, float)) and MIN_VALID_BUDGET <= budget <= MAX_VALID_BUDGET:
                budget_adj = adjust_for_inflation(budget, movie["release_date"], INFLATION_LOOKUP)
                valid_budgets.append(int(budget_adj) if budget_adj else int(budget))
        
        avg_budget_L5 = sum(valid_budgets) / len(valid_budgets) if valid_budgets else 0
        
        # Director collaboration
        director_collab_count = 0
        if current_director_id:
            director_movies = self.get_director_filmography(current_director_id)
            director_collab_count = len(company_movie_ids & director_movies)
        
        # Top 3 genres
        top_3_genres = None
        if genre_counter:
            named_counts = {}
            for gid, count in genre_counter.items():
                gname = GENRE_MAP.get(gid)
                if gname:
                    named_counts[gname] = named_counts.get(gname, 0) + count
            if named_counts:
                sorted_genres = sorted(named_counts.items(), key=lambda x: x[1], reverse=True)
                top_3_genres = "|".join(g for g, _ in sorted_genres[:3])
        
        # Average movies per year
        avg_movies_per_year = 0
        if company_movies:
            years = [m["release_date"].year for m in company_movies]
            if years:
                year_span = max(years) - min(years) + 1
                avg_movies_per_year = len(company_movies) / year_span if year_span > 0 else len(company_movies)
        
        return {
            "num_prior_films": len(company_movies),
            "prior_films_in_genre": prior_films_in_genre,
            "prior_franchise_films": prior_franchise_films,
            "is_major_studio": 1 if is_major_studio(company_name) else 0,
            "is_rebranded_studio": 1 if used_inheritance else 0,
            "director_collab_count": director_collab_count,
            "top_3_genres": top_3_genres,
            "avg_budget_L5": round(avg_budget_L5, 2),
            "avg_movies_per_year": round(avg_movies_per_year, 2),
        }

    # -------- Per-movie processing --------
    def _process_single_movie(self, row_data):
        """Process a single movie for parallel execution. Thread-safe."""
        movie_id, date_id, movie_title = row_data
        
        try:
            target_date = pd.to_datetime(str(date_id), format='%Y%m%d')
        except Exception:
            return None, f"Movie {movie_id}: SKIP (invalid date)"
        
        # Target movie details via central cache
        movie_details = self.get_movie_details_cached(movie_id)
        
        target_genres = set(g['id'] for g in movie_details.get('genres', [])) if movie_details else set()
        collection_info = movie_details.get('belongs_to_collection') if movie_details else None
        collection_id = collection_info.get('id') if collection_info else None
        
        # Franchise info (empty if not in collection)
        _, franchise_movie_producers = self.get_collection_info(collection_id, target_date)
        
        current_director_id = self.get_director_for_movie(movie_id)
        companies = self.get_production_companies(movie_id)
        
        # Deduplicate companies efficiently
        seen_ids = set()
        companies_list = []
        for comp in companies:
            comp_id = comp.get('id')
            if comp_id and comp_id not in seen_ids:
                seen_ids.add(comp_id)
                companies_list.append({'id': comp_id, 'name': comp.get('name', '')})
        
        num_companies = len(companies_list)
        
        # Handle movies with no production companies
        if not companies_list:
            result = {
                'movie_id': movie_id,
                'date_id': date_id,
                'prod_num_companies': 0,
                'all_production_companies': None,
                'all_company_ids': None,
                'prod_has_major_company': 0,
                'prod_is_rebranded_studio': 0,
                'prod_studio_group': None,
                'lead_company': None,
                'prod_num_prior_films_lead': 0,
                'prod_num_prior_films_total': 0,
                'prod_avg_budget_L5_lead': 0,
                'top_3_genres': None,
                'prod_has_prior_in_genre': 0,
                'prod_prior_films_in_genre_total': 0,
                'prod_prior_films_in_this_franchise_total': 0,
                'prod_director_collab_count_total': 0,
                'prod_avg_movies_per_year': 0,
            }
            return result, f"Movie {movie_id}: ✓ (no production companies)"
        
        # Pre-fetch filmographies for lead selection
        for comp in companies_list:
            self.get_company_filmography(comp['id'], target_date=None)
        with self._cache_lock:
            filmography_cache_copy = {}
            for (cid, tdate), movies in self.company_filmography_cache.items():
                if tdate is None:
                    filmography_cache_copy[cid] = movies
        
        # Snapshot of filmography cache for consistent lead selection
        with self._cache_lock:
            filmography_cache_copy = dict(self.company_filmography_cache)
        lead_company, lead_company_id = get_lead_company(companies_list, filmography_cache_copy)
        prod_studio_group = get_studio_group(lead_company) if lead_company else None
        
        # Build profiles for all companies
        company_profiles = []
        lead_profile = None
        for comp in companies_list:
            profile = self.get_company_profile(
                comp['id'], comp['name'], target_date,
                franchise_movie_producers, target_genres, current_director_id
            )
            company_profiles.append(profile)
            if comp['id'] == lead_company_id:
                lead_profile = profile
        
        if not lead_profile and company_profiles:
            lead_profile = company_profiles[0]
        
        # Aggregate features
        prod_has_major_company = 1 if any(p['is_major_studio'] == 1 for p in company_profiles) else 0
        prod_is_rebranded_studio = 1 if any(p.get('is_rebranded_studio', 0) == 1 for p in company_profiles) else 0
        prod_num_prior_films_total = sum(p['num_prior_films'] for p in company_profiles)
        prod_prior_films_in_genre_total = sum(p['prior_films_in_genre'] for p in company_profiles)
        prod_prior_films_in_this_franchise_total = sum(p['prior_franchise_films'] for p in company_profiles)
        prod_director_collab_count_total = sum(p['director_collab_count'] for p in company_profiles)
        prod_has_prior_in_genre = 1 if prod_prior_films_in_genre_total > 0 else 0
        
        prod_num_prior_films_lead = lead_profile['num_prior_films'] if lead_profile else 0
        prod_avg_budget_L5_lead = lead_profile['avg_budget_L5'] if lead_profile else 0
        top_3_genres = lead_profile['top_3_genres'] if lead_profile else None
        
        # Average movies per year across companies
        movies_per_year_values = [p['avg_movies_per_year'] for p in company_profiles if p['avg_movies_per_year'] > 0]
        prod_avg_movies_per_year = round(sum(movies_per_year_values) / len(movies_per_year_values), 2) if movies_per_year_values else 0
        
        result = {
            'movie_id': movie_id,
            'date_id': date_id,
            'prod_num_companies': num_companies,
            'all_production_companies': '|'.join([c['name'] for c in companies_list]) if companies_list else None,
            'all_company_ids': '|'.join([str(c['id']) for c in companies_list]) if companies_list else None,
            'prod_has_major_company': prod_has_major_company,
            'prod_is_rebranded_studio': prod_is_rebranded_studio,
            'prod_studio_group': prod_studio_group,
            'lead_company': lead_company,
            'prod_num_prior_films_lead': prod_num_prior_films_lead,
            'prod_num_prior_films_total': prod_num_prior_films_total,
            'prod_avg_budget_L5_lead': prod_avg_budget_L5_lead,
            'top_3_genres': top_3_genres,
            'prod_has_prior_in_genre': prod_has_prior_in_genre,
            'prod_prior_films_in_genre_total': prod_prior_films_in_genre_total,
            'prod_prior_films_in_this_franchise_total': prod_prior_films_in_this_franchise_total,
            'prod_director_collab_count_total': prod_director_collab_count_total,
            'prod_avg_movies_per_year': prod_avg_movies_per_year,
        }
        
        lead_display = lead_company[:25] if lead_company and len(lead_company) > 25 else lead_company
        return result, f"Movie {movie_id}: ✓ (lead: {lead_display})"

    # -------- Dataset builder --------
    def build_dataset(self):
        if not os.path.exists(INPUT_FILE):
            print(f"❌ {INPUT_FILE} not found. Run Fact Table Generator first.")
            return pd.DataFrame()
        
        # Load checkpoint
        existing_df, processed_ids = load_checkpoint(OUTPUT_FILE, id_col='movie_id')
        
        # Load input data
        fact_df = pd.read_csv(INPUT_FILE)
        
        print(f"🎬 Processing Production Company Features for {len(fact_df)} movies (PARALLEL with {self.max_workers} workers)...")
        if processed_ids:
            print(f"   ⏭️ Skipping {len(processed_ids)} already processed movies")
        
        if fact_df.empty:
            return pd.DataFrame()
        
        # Prepare items for parallel processing (skip already processed)
        items_to_process = []
        for idx, row in fact_df.iterrows():
            movie_id = int(row['movie_id'])
            date_id = int(row['date_id'])
            
            # Skip invalid dates
            if date_id == 99991231:
                continue
            
            # Skip if already processed
            if is_already_processed(movie_id, processed_ids):
                continue
            
            movie_title = row.get('title', f'Movie {movie_id}')
            items_to_process.append((movie_id, date_id, movie_title))
        
        if not items_to_process:
            print("✅ All movies already processed!")
            return existing_df
        
        print(f"   📋 {len(items_to_process)} movies to process")
        
        # Run parallel processing
        results_list = parallel_process(
            items=items_to_process,
            process_func=self._process_single_movie,
            max_workers=self.max_workers,
            desc="Production History"
        )
        
        # Collect successful results
        output_rows = []
        success_count = 0
        
        for result, message in results_list:
            if result is not None:
                output_rows.append(result)
                success_count += 1
        
        # Save all results using checkpoint system
        if output_rows:
            existing_df = save_checkpoint(output_rows, existing_df, OUTPUT_FILE)
        
        print(f"\n🎉 Completed! {OUTPUT_FILE} saved")
        print(f"   Total: {len(existing_df)} production company records")
        print(f"   This run: {success_count} processed")
        
        # Show sample
        if not existing_df.empty:
            print("\nSample:")
            display_cols = ['movie_id']
            for col in ['lead_company', 'prod_num_companies', 'prod_has_major_company', 'prod_num_prior_films_total']:
                if col in existing_df.columns:
                    display_cols.append(col)
            print(existing_df[display_cols].head())
        
        return existing_df

# Initialize and run
builder = ProductionHistoryBuilder(max_workers=20 )
prod_df = builder.build_dataset()

In [ ]:
# ============================================================
# AVATAR PREDICTION DATASET MERGER 
# ============================================================
logging.basicConfig(format='[%(asctime)s] %(levelname)s: %(message)s', level=logging.INFO)

def create_prediction_dataset():
    logging.info("🎬 Starting AVATAR Prediction Dataset Merger")

    # ==========================================
    # 1. DEFINE CSV FILES AND COLUMNS TO DROP
    # ==========================================
    
    csv_config = {
        "Movie_Dimension.csv": {
            "suffix": "_movie",
            # Actual columns: movie_id, title, imdb_id, original_language, genres, genre_count,
            # production_countries, country_count, is_us_film, mpaa_rating, is_based_on_source,
            # num_competing_movies, date_id, release_date, collection_id, budget_raw, budget_adjusted,
            # budget_class, revenue_first_week_raw, revenue_first_week_adjusted, 
            # revenue_opening_weekend_raw, revenue_opening_weekend_adjusted, first_week_days_collected,
            # opening_theater_count, runtime_minutes
            "drop_cols": [
                "movie_id", "imdb_id", "date_id", "collection_id",
                "first_week_days_collected"
            ],
            "merge_key": "movie_id"
        },
        "Dim_Franchise.csv": {
            "suffix": "_franchise",
            # Actual columns: movie_id, collection_id, is_franchise, is_shared_universe,
            # franchise_prior_movies_count, franchise_movies_missing_data,
            # franchise_avg_prior_first_week_adj, franchise_sum_prior_first_week_adj,
            # franchise_max_prior_first_week_adj, franchise_avg_prior_opening_weekend_adj,
            # franchise_sum_prior_opening_weekend_adj, franchise_max_prior_opening_weekend_adj,
            # franchise_last_film_first_week_adj, franchise_last_film_opening_weekend_adj,
            # years_since_last_franchise_film, L1_title, L1_release_date, L1_first_week_adj,
            # L1_opening_weekend_adj, ... L10_title, L10_release_date, L10_first_week_adj, L10_opening_weekend_adj
            "drop_cols": [
                "movie_id", "collection_id",
                # Raw values
                "franchise_avg_prior_first_week_raw", "franchise_sum_prior_first_week_raw", "franchise_max_prior_first_week_raw", "franchise_last_film_first_week_raw", "franchise_avg_prior_opening_weekend_raw", 
                "franchise_sum_prior_opening_weekend_raw", "franchise_max_prior_opening_weekend_raw", "franchise_last_film_opening_weekend_raw", "franchise_avg_prior_budget_raw", "franchise_max_prior_budget_raw",
                "franchise_last_film_budget_raw", "L1_first_week_raw", "L1_opening_weekend_raw", "L2_first_week_raw", "L2_opening_weekend_raw", "L3_first_week_raw", "L3_opening_weekend_raw", "L4_first_week_raw", 
                "L4_opening_weekend_raw", "L5_first_week_raw", "L5_opening_weekend_raw", "L6_first_week_raw", "L6_opening_weekend_raw", "L7_first_week_raw", "L7_opening_weekend_raw", "L8_first_week_raw", 
                "L8_opening_weekend_raw", "L9_first_week_raw", "L9_opening_weekend_raw", "L10_first_week_raw", "L10_opening_weekend_raw", "director_avg_prior_budget_raw", "director_budget_coverage_pct_director",  
                # L1-L10 detail columns (too granular for ML)
                "L1_title", "L1_release_date", "L1_first_week_adj", "L1_opening_weekend_adj",
                "L2_title", "L2_release_date", "L2_first_week_adj", "L2_opening_weekend_adj",
                "L3_title", "L3_release_date", "L3_first_week_adj", "L3_opening_weekend_adj",
                "L4_title", "L4_release_date", "L4_first_week_adj", "L4_opening_weekend_adj",
                "L5_title", "L5_release_date", "L5_first_week_adj", "L5_opening_weekend_adj",
                "L6_title", "L6_release_date", "L6_first_week_adj", "L6_opening_weekend_adj",
                "L7_title", "L7_release_date", "L7_first_week_adj", "L7_opening_weekend_adj",
                "L8_title", "L8_release_date", "L8_first_week_adj", "L8_opening_weekend_adj",
                "L9_title", "L9_release_date", "L9_first_week_adj", "L9_opening_weekend_adj",
                "L10_title", "L10_release_date", "L10_first_week_adj", "L10_opening_weekend_adj",
            ],
            # KEEPING: is_franchise, is_shared_universe, franchise_prior_movies_count,
            # franchise_movies_missing_data, franchise_avg/sum/max/last_opening_weekend_adj,
            #  franchise_avg/sum/max/last_first_week_adj, years_since_last_franchise_film
            "merge_key": "movie_id"
        },
        "Dim_Director.csv": {
            "suffix": "_director",
            # Actual columns: movie_id, num_directors, director_id, director_name, other_directors,
            # total_films_before, years_active, years_since_last_film, avg_runtime_L10,
            # director_avg_prior_budget_raw/adj, director_films_with_budget, director_films_missing_budget,
            # director_budget_coverage_pct, director_prior_franchise_films, top_3_genres,
            # top_5_cast_collaborators, PLUS 12 NEW BOX OFFICE FEATURES (Raw/Adj)
            "drop_cols": [
                "movie_id", "num_directors", "director_id", 
                "director_name", "other_directors",
            ],
            # KEEPING: total_films_before, years_active, years_since_last_film, avg_runtime_L10,
            # director_avg_prior_budget_adj, director_budget_coverage_pct, director_prior_franchise_films, 
            # top_3_genres, top_5_cast_collaborators
            "merge_key": "movie_id"
        },
        "Dim_Cast.csv": {
            "suffix": "_cast",
            # Actual columns: movie_id, date_id, total_cast_size, avg_costar_affinity,
            # actors_worked_with_director, avg_films_with_director, total_films_with_director,
            # avg_years_active, top_3_genres, avg_films_per_actor, avg_budget_L10,
            # returning_franchise_actors, avg_genre_fit
            "drop_cols": ["movie_id", "date_id"],
            # KEEPING: total_cast_size, avg_costar_affinity, actors_worked_with_director,
            # avg_films_with_director, total_films_with_director, avg_years_active, top_3_genres,
            # avg_films_per_actor, avg_budget_L10, returning_franchise_actors, avg_genre_fit
            "merge_key": "movie_id"
        },
        "Dim_Date.csv": {
            "suffix": "_date",
            # Actual columns: date_id, date_iso, year, quarter, month, month_name, day_of_month,
            # day_of_week, day_name, is_weekend, is_summer_season, is_holiday_season,
            # is_pandemic_release, is_holiday_release
            "drop_cols": ["date_id", "date_iso", "month_name", "day_name"],
            # KEEPING: year, quarter, month, day_of_month, day_of_week, is_weekend,
            # is_summer_season, is_holiday_season, is_pandemic_release, is_holiday_release
            "merge_key": "date_id"
        },
        "Dim_Production_Company.csv": {
            "suffix": "_prod",
            # Actual columns: movie_id, date_id, prod_num_companies, all_production_companies,
            # all_company_ids, prod_has_major_company, prod_is_rebranded_studio, prod_studio_group,
            # lead_company, prod_num_prior_films_lead, prod_num_prior_films_total, 
            # prod_avg_budget_L5_lead, top_3_genres, prod_has_prior_in_genre,
            # prod_prior_films_in_genre_total, prod_prior_films_in_this_franchise_total,
            # prod_director_collab_count_total, prod_avg_movies_per_year
            "drop_cols": [
                "movie_id", "date_id", "all_production_companies",
                "all_company_ids", "prod_studio_group", "prod_avg_budget_L5_lead"
            ],
            # KEEPING: prod_num_companies, prod_has_major_company, prod_is_rebranded_studio,
            # lead_company, prod_num_prior_films_lead, prod_num_prior_films_total, top_3_genres,
            # prod_has_prior_in_genre, prod_prior_films_in_genre_total,
            # prod_prior_films_in_this_franchise_total, prod_director_collab_count_total,
            # prod_avg_movies_per_year
            "merge_key": "movie_id"
        }
    }

    # ==========================================
    # 2. LOAD AND VALIDATE ALL CSVs
    # ==========================================
    
    # Check all files exist
    missing_files = [f for f in csv_config.keys() if not os.path.exists(f)]
    if missing_files:
        logging.error(f"❌ Missing CSV files: {missing_files}")
        logging.error("Please run the corresponding builder cells first.")
        return None

    dataframes = {}
    for csv_file, config in csv_config.items():
        try:
            df = pd.read_csv(csv_file)
            dataframes[csv_file] = df
            logging.info(f"✅ Loaded {csv_file}: {len(df)} rows × {len(df.columns)} cols")
        except Exception as e:
            logging.error(f"❌ Failed to load {csv_file}: {e}")
            return None

    # ==========================================
    # 3. START WITH MOVIE_DIMENSION AS BASE
    # ==========================================
    
    base_csv = "Movie_Dimension.csv"
    df_base = dataframes[base_csv].copy()
    
    # Store merge keys before processing
    movie_id_col = df_base["movie_id"].copy()
    date_id_col = df_base["date_id"].copy() if "date_id" in df_base.columns else None
    
    logging.info(f"\n📌 Base table: {base_csv} with {len(df_base)} movies")
    logging.info(f"   movie_id range: {movie_id_col.min()} - {movie_id_col.max()}")
    
    # ==========================================
    # 4. PROCESS EACH CSV: DROP COLS + ADD SUFFIX
    # ==========================================
    
    processed_dfs = {}
    
    for csv_file, config in csv_config.items():
        df = dataframes[csv_file].copy()
        suffix = config["suffix"]
        drop_cols = config["drop_cols"]
        merge_key = config["merge_key"]
        
        # Store merge key before dropping
        if merge_key in df.columns:
            merge_key_values = df[merge_key].copy()
        else:
            logging.error(f"❌ {csv_file}: Missing merge key '{merge_key}'")
            return None
        
        # Drop specified columns (only those that exist)
        cols_to_drop = [c for c in drop_cols if c in df.columns]
        missing_drops = [c for c in drop_cols if c not in df.columns]
        if missing_drops:
            logging.warning(f"   {csv_file}: Columns not found (skipping): {missing_drops}")
        
        df = df.drop(columns=cols_to_drop)
        
        # Add suffix to remaining columns
        df.columns = [f"{col}{suffix}" for col in df.columns]
        
        # Add back the merge key (without suffix for joining)
        df[merge_key] = merge_key_values
        
        # Handle duplicates (keep first for dimensions)
        if csv_file != base_csv:
            dups = df.duplicated(subset=[merge_key], keep='first').sum()
            if dups > 0:
                logging.warning(f"   {csv_file}: Removing {dups} duplicate {merge_key}s")
                df = df.drop_duplicates(subset=[merge_key], keep='first')
        
        processed_dfs[csv_file] = df
        
        kept_cols = len(df.columns) - 1  # Subtract merge key
        dropped_count = len(cols_to_drop)
        logging.info(f"   {csv_file}: Dropped {dropped_count} cols, kept {kept_cols} (+ merge key)")
    
    # ==========================================
    # 5. MERGE ALL TABLES
    # ==========================================
    
    logging.info("\n🔗 Merging all dimension tables...")
    
    # Start with base (Movie_Dimension)
    final_df = processed_dfs[base_csv].copy()
    base_rows = len(final_df)
    logging.info(f"   Starting with {base_csv}: {base_rows} rows")
    
    # Merge order
    merge_order = [
        ("Dim_Franchise.csv", "movie_id"),
        ("Dim_Director.csv", "movie_id"),
        ("Dim_Cast.csv", "movie_id"),
        ("Dim_Production_Company.csv", "movie_id"),
        ("Dim_Date.csv", "date_id"),
    ]
    
    for csv_file, merge_key in merge_order:
        df_to_merge = processed_dfs[csv_file]
        
        # For date merge, we need date_id in final_df
        if merge_key == "date_id" and merge_key not in final_df.columns:
            # Add date_id back from original base
            if date_id_col is not None:
                final_df["date_id"] = date_id_col.values
            else:
                logging.warning(f"   ⚠️ Skipping {csv_file}: No date_id in base table")
                continue
        
        before_rows = len(final_df)
        final_df = final_df.merge(df_to_merge, on=merge_key, how="left")
        
        if len(final_df) != before_rows:
            logging.warning(f"   ⚠️ {csv_file} merge changed row count: {before_rows} → {len(final_df)}")
        else:
            # Count successful matches
            suffix = csv_config[csv_file]["suffix"]
            sample_col = [c for c in final_df.columns if c.endswith(suffix) and c != merge_key]
            if sample_col:
                matched = final_df[sample_col[0]].notna().sum()
                logging.info(f"   ✅ Merged {csv_file}: {matched}/{len(final_df)} matched")
    
    # ==========================================
    # 6. CLEANUP AND FINAL CHECKS
    # ==========================================
    
    # Remove date_id 
    if "date_id" in final_df.columns:
        final_df = final_df.drop(columns=["date_id"])
    # Remove movie_id
    if "movie_id" in final_df.columns:
        final_df = final_df.drop(columns=["movie_id"])
    
    logging.info(f"\n✅ Merge complete! Final shape: {final_df.shape}")
    
    # ==========================================
    # 7. DATA QUALITY SUMMARY
    # ==========================================
    
    logging.info("\n📋 Data Quality Summary:")
    logging.info(f"   Total rows: {len(final_df)}")
    logging.info(f"   Total columns: {len(final_df.columns)}")
    
    # Check nulls per source
    for csv_file, config in csv_config.items():
        suffix = config["suffix"]
        suffix_cols = [c for c in final_df.columns if c.endswith(suffix)]
        if suffix_cols:
            null_pct = final_df[suffix_cols].isna().mean().mean() * 100
            logging.info(f"   {suffix} columns ({len(suffix_cols)}): {null_pct:.1f}% nulls")
    
    # Overall stats
    total_nulls = final_df.isna().sum().sum()
    total_cells = len(final_df) * len(final_df.columns)
    null_pct = (total_nulls / total_cells * 100) if total_cells > 0 else 0
    logging.info(f"   Overall null rate: {null_pct:.2f}%")
    
    memory_mb = final_df.memory_usage(deep=True).sum() / 1024 / 1024
    logging.info(f"   Memory usage: {memory_mb:.2f} MB")

    # ==========================================
    # 8. SAVE MERGED DATASET
    # ==========================================
    
    output_file = "avatar_prediction_dataset.csv"
    final_df.to_csv(output_file, index=False)
    
    file_size_mb = os.path.getsize(output_file) / 1024 / 1024
    logging.info(f"\n✅ Saved: {output_file} ({file_size_mb:.2f} MB)")
    logging.info(f"🎉 Pipeline complete! {len(final_df)} rows × {len(final_df.columns)} columns")

    # Preview
    print("\n" + "="*80)
    print("AVATAR PREDICTION DATASET - Column Summary")
    print("="*80)
    
    for csv_file, config in csv_config.items():
        suffix = config["suffix"]
        suffix_cols = sorted([c for c in final_df.columns if c.endswith(suffix)])
        print(f"\n{suffix} ({len(suffix_cols)} columns):")
        for col in suffix_cols:
            print(f"   • {col}")
    
    print("\n" + "="*80)
    print(f"\nFull column list ({len(final_df.columns)} total):")
    print(final_df.columns.tolist())
    
    print("\n" + "="*80)
    print(f"Sample (first 3 rows):")
    print(final_df.head(3).T)  # Transposed for readability
    
    return final_df

# Run the merger
merged_df = create_prediction_dataset()